# Does NCA pre-pre-training accelerate subsequent language learning?

**Version 4: exact-prefix controls and fresh paired replication.**
The objective is faster language learning through pre-pre-training. Seven
initializations enter the SAME language stage: scratch, grid NCA, graph NCA,
the two original shuffled controls, and two stronger prefix-matched controls.
Those new controls preserve initial conditioning, supervised-prefix patch
frequencies and marker formatting while scrambling later patch order.

Select one GPU accelerator and enable Internet before Run All. Six fresh
cohorts use 1000 PPT updates (zero for scratch) plus 2000 equal language updates.
The transformer, rule filter, topology and serializer retain v3 settings.
The official test stays untouched. This is a prospective validation study,
not confirmation on a new language task. No positive outcome is guaranteed.

The co-primary comparisons are graph versus grid and graph versus the new
prefix-matched graph control, at the SAME paired scratch-endpoint loss.
All six cohorts stay in the analysis, including ties and missed targets.
Two one-sided sign tests use alpha .025 each; six strict wins are required.
The protocol/config/source identities are archived before benchmark training.

Allow roughly **7-8 hours on a comparable T4, plus verification/export overhead**:
this extrapolates the returned v3 run, not a new timing measurement. Disposable
calibration on your GPU prints an estimate. The final ZIP may be several GB
because it retains all stage checkpoints and six real starting-weight snapshots.
If a structured synthetic gate fails, the language suite is skipped and the
diagnostics still export. Passing a gate or finishing does not prove acceleration.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
ROOT = Path('/kaggle/working') / f'nca_transfer_v4_{RUN_ID}'
ROOT.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
os.environ['TIKTOKEN_CACHE_DIR'] = str(ROOT / 'tokenizer_cache')
os.environ['MPLCONFIGDIR'] = str(ROOT / 'plot_cache')
print('Run:', RUN_ID, 'Workspace:', ROOT)

for directory in ['configs', 'docs', 'sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431', 'tests', 'transfer_v4']:
    Path(directory).mkdir(parents=True, exist_ok=True)


## Embedded, reviewable sources
These files match the locally checked v4 source. No Git checkout is required.

In [ ]:
%%writefile transfer_v4/__init__.py
"""Independent-stage transfer with exact-prefix controls and fresh replication (v4)."""


In [ ]:
%%writefile transfer_v4/analysis.py
"""Frozen common-target comparisons; never discard ties or censored cohorts."""
import math
import statistics

from .common import STUDY


def first_hit(rows, threshold):
    return next((r for r in rows if r["validation_nll"] <= threshold), None)


def common_target_analysis(curves, c):
    comparisons = []
    keys = ("language_tokens", "language_seconds", "cumulative_seconds")
    secondary_pairs = (("prefix_grid", "grid_nca"), ("shuffled_graph", "graph_nca"),
                       ("shuffled_nca", "grid_nca"), ("scratch", "graph_nca"))
    for reference_arm, arm in list(STUDY["primary_pairs"])+list(secondary_pairs):
        rows = []
        for seed in c["seeds"]:
            scratch = curves.get(("scratch", seed), [])
            reference = curves.get((reference_arm, seed), [])
            treatment = curves.get((arm, seed), [])
            threshold = scratch[-1]["validation_nll"] if scratch and scratch[-1]["language_step"] == c["language"]["steps"] else None
            a = first_hit(reference, threshold) if reference and threshold is not None else None
            b = first_hit(treatment, threshold) if treatment and threshold is not None else None
            row = {"seed": seed, "target_nll": threshold,
                   "reference_reached": a is not None, "graph_reached": b is not None,
                   "reference_step": a["language_step"] if a else None,
                   "graph_step": b["language_step"] if b else None}
            for key in keys:
                # A step-zero target does not define a positive token ratio.
                row[key+"_speedup"] = a[key]/b[key] if a and b and a[key] > 0 and b[key] > 0 else None
            rows.append(row)
        item = {"reference_arm": reference_arm, "arm": arm, "rows": rows,
                "target_policy": STUDY["target"], "declared_cohorts": len(c["seeds"])}
        for key in keys:
            values = [r[key+"_speedup"] for r in rows]
            complete = bool(values) and all(v is not None and v > 0 for v in values)
            item["geomean_"+key+"_speedup_all_cohorts"] = math.exp(statistics.mean(math.log(v) for v in values)) if complete else None
        ratios = [r["language_tokens_speedup"] for r in rows]
        complete = bool(ratios) and all(v is not None for v in ratios)
        wins = sum(v is not None and v > 1 for v in ratios)
        n = len(rows)
        # Ties count as non-wins (conservative). Missing/censored: no p-value.
        p = sum(math.comb(n, k) for k in range(wins, n+1))/2**n if complete else None
        item.update(strict_token_wins=wins, ties_count_as_nonwins=True,
                    one_sided_sign_p=p, alpha=STUDY["alpha_per_comparison"],
                    criterion_met=bool(complete and n >= STUDY["minimum_cohorts"]
                                       and p <= STUDY["alpha_per_comparison"]
                                       and item["geomean_language_tokens_speedup_all_cohorts"] > 1))
        comparisons.append(item)
    primary, secondary = comparisons[:2], comparisons[2:]
    for item in secondary:
        for key in ("one_sided_sign_p", "alpha", "criterion_met"):
            item.pop(key)
        item["scope"] = "Descriptive secondary comparison; no additional significance claim"
    eligible = c["purpose"] == "development"
    return {"study": STUDY, "comparisons": primary,
            "secondary_common_target_comparisons": secondary,
            "eligible_for_research_interpretation": eligible,
            "supports_graph_advantage_on_declared_target": eligible and all(i["criterion_met"] for i in primary),
            "multiplicity": "Two co-primary one-sided sign tests, alpha .025 each (Bonferroni family bound .05)",
            "scope": "Language-token transfer advantage on an inspected validation task; not end-to-end GPU speed or untouched confirmation",
            "censoring": "Any missing crossing withholds the all-cohort ratio and significance claim; no imputation or success-only subset"}


In [ ]:
%%writefile transfer_v4/calibrate.py
"""Disposable on-device timing calibration; never changes a research model."""
import hashlib
from pathlib import Path
import statistics
import time

import numpy as np
import torch

from .common import ARMS, dump, sync
from .model import LM, language_model
from .training import evaluate, optimizer_for, update


def calibrate(c, arrays, manifest, device, output):
    values = {}
    for name, synthetic in (("ppt", True), ("language", False)):
        stage = c[name]
        model = (LM(c["model"], c["nca"]["states"]**4+2, 0, 1) if synthetic else
                 language_model(c, manifest["text"]["vocab"], 0)).to(device)
        optimizer = optimizer_for(model, stage, synthetic)
        array = arrays["nca_train"] if synthetic else arrays["text_train"]
        validation = arrays["nca_validation"] if synthetic else arrays["text_validation"]
        rng = np.random.default_rng(0)
        order_hash = hashlib.sha256()
        measurements = []
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        for step in range(4):
            sync(device)
            started = time.perf_counter()
            update(model, optimizer, array, c, stage, min(step, stage["steps"]-1), rng, device, synthetic, order_hash)
            sync(device)
            if step:
                measurements.append(time.perf_counter()-started)
        started = time.perf_counter()
        evaluate(model, validation, c, device, synthetic)
        sync(device)
        values[name] = {"seconds_per_update_median": statistics.median(measurements),
                        "validation_seconds": time.perf_counter()-started,
                        "parameters": sum(p.numel() for p in model.parameters()),
                        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated()/1e6 if device == "cuda" else None}
        del model, optimizer
    count = len(c["seeds"])
    approx = 0.
    for name, jobs in (("ppt", (len(ARMS)-1)*count), ("language", len(ARMS)*count)):
        steps = c[name]["steps"]
        evals = (steps+c["eval_every"]-1)//c["eval_every"]+1
        approx += jobs*(steps*values[name]["seconds_per_update_median"]+evals*values[name]["validation_seconds"])
    result = {"purpose": "Disposable warmup/timing calibration only; no checkpoint used for transfer",
              "device": device, "stage_measurements": values,
              "approx_full_training_and_validation_minutes": approx/60,
              "excludes": "Generation, checkpoint saves, verification, plotting, downloads, queueing and changing GPU load; not a runtime guarantee"}
    dump(Path(output)/"calibration.json", result)
    print(f"Measured estimate for full training + validation: {approx/60:.1f} minutes, excluding preparation/checkpoint/verification/archive work.", flush=True)
    return result


In [ ]:
%%writefile transfer_v4/common.py
from __future__ import annotations

import hashlib
import json
import math
from pathlib import Path
import random

import numpy as np
import torch

ARMS = ("scratch", "grid_nca", "graph_nca", "shuffled_nca", "shuffled_graph", "prefix_grid", "prefix_graph")
SYNTHETIC_ARMS = ARMS[1:]
DATA_KEYS = {"grid_nca": "nca", "graph_nca": "graph",
             "shuffled_nca": "shuffled", "shuffled_graph": "shuffled_graph",
             "prefix_grid": "prefix_grid", "prefix_graph": "prefix_graph"}
REFERENCE_COMMIT = "bdd1c71e04eb2e5a2c91481faf278e95846b4431"
STUDY = {"id": "prefix_frequency_control_v1", "target": "paired_scratch_endpoint",
         "primary_pairs": [["grid_nca", "graph_nca"], ["prefix_graph", "graph_nca"]],
         "alpha_per_comparison": .025, "minimum_cohorts": 6,
         "control": "initial_frame_and_supervised_prefix_histogram",
         "crossing": "first_recorded_no_interpolation_all_cohorts_required"}


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False)+"\n", encoding="utf-8")
    temporary.replace(path)


def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()


def state_sha(state):
    h = hashlib.sha256()
    for name, tensor in sorted(state.items()):
        h.update(name.encode())
        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def source_hashes():
    root = Path(__file__).resolve().parent
    return {p.name: sha_file(p) for p in sorted(root.glob("*.py"))}


def sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_config(path):
    c = json.loads(Path(path).read_text(encoding="utf-8"))
    validate_config(c)
    return c


def validate_config(c):
    if c["schema_version"] != 4:
        raise ValueError("Expected a mechanism-transfer v4 configuration")
    if c["purpose"] not in ("development", "execution_check"):
        raise ValueError("Unknown run purpose")
    if not c["seeds"] or len(set(c["seeds"])) != len(c["seeds"]):
        raise ValueError("Model seeds must be unique")
    m, n = c["model"], c["nca"]
    if m["width"] % m["heads"] or (m["width"]//m["heads"]) % 2:
        raise ValueError("RoPE requires an even head dimension")
    if n["grid"] % 2 or n["states"] < 2:
        raise ValueError("Expected an even grid and a categorical alphabet")
    if n["states"]**4+2 > 65536:
        raise ValueError("Synthetic token IDs must fit uint16")
    if m["context"] < 2*frame_length(c):
        raise ValueError("Context must contain at least two complete frames")
    if any(c[stage]["steps"] <= 0 for stage in ("ppt", "language")):
        raise ValueError("PPT and language budgets are separate positive lengths")
    if not 0 <= n["filter_low"] < n["filter_high"]:
        raise ValueError("Invalid complexity band")
    if c["purpose"] == "development" and not c["gate"]["enforce"]:
        raise ValueError("Development runs must enforce the held-out synthetic gate")
    if c["purpose"] == "development" and c["text"]["tokenizer"] != "gpt2":
        raise ValueError("Byte fixtures are restricted to execution checks")
    if c["precision"] != "fp32":
        raise ValueError("This audited version uses fp32 in both stages")
    if n["grid"] < 4 or not 0 <= c["graph"]["rewire_fraction"] <= 1:
        raise ValueError("Graph needs side >=4 and a rewiring fraction in [0,1]")
    if c["purpose"] == "development" and c["graph"]["rewire_fraction"] <= 0:
        raise ValueError("Development graph treatment must change topology")
    if c["graph"]["rule_selection"] != "intersection" or c["graph"]["replication"] != "independent_cache_per_model_seed":
        raise ValueError("Use paired intersection filtering and independent replicate caches")
    if c.get("study") != STUDY:
        raise ValueError("The v4 study criterion is frozen; declare a new version to change it")
    if c["purpose"] == "development":
        if len(c["seeds"]) < STUDY["minimum_cohorts"] or set(c["seeds"]) & {17, 29, 43}:
            raise ValueError("Use at least six fresh cohorts, excluding the inspected v3 seeds")
        if c["text"]["evaluate_test"]:
            raise ValueError("V4 is a validation-stage replication/mechanism study; reserve the official test")
    if any(c[key] <= 0 for key in ("eval_every", "eval_batch", "batch", "accumulate", "ppt_eval_rules")):
        raise ValueError("Batch and evaluation counts must be positive")


def frame_length(c):
    return (c["nca"]["grid"]//2)**2+2


def effective_tokens(c):
    return c["batch"]*c["accumulate"]*c["model"]["context"]


def lr_at(step, stage):
    length = stage["steps"]
    warm = max(1, min(length, round(length*stage["warmup_fraction"])))
    if step < warm:
        return stage["lr"]*(step+1)/warm
    fraction = (step-warm)/max(1, length-warm-1)
    return stage["lr"]*(.1+.9*(1+math.cos(math.pi*fraction))/2)


In [ ]:
%%writefile transfer_v4/controls.py
"""Matched scrambling controls for the actual synthetic training objective.

Each trajectory keeps its initial conditioning frame, marker values/positions,
and supervised-prefix patch histogram. Only subsequent patch order is permuted.
The unused suffix is shuffled separately so no token enters/leaves the prefix.
This preserves per-rule frequency information; it is not an IID-noise control.
"""
from __future__ import annotations

import numpy as np

from .common import frame_length


def prefix_shuffled(tokens, c, seed):
    result = tokens.copy()
    boundary = c["model"]["context"]+1
    first = frame_length(c)
    if tokens.ndim != 2 or tokens.shape[1] < boundary:
        raise ValueError("Trajectories must contain the complete trained prefix")
    rng = np.random.default_rng(seed)
    cutoff = c["nca"]["states"]**4
    for row in result:
        for start, end in ((first, boundary), (boundary, len(row))):
            positions = np.arange(start, end)
            positions = positions[row[positions] < cutoff]
            row[positions] = rng.permutation(row[positions])
    return result


def validate_prefix_control(original, control, c):
    if original.shape != control.shape or original.dtype != control.dtype:
        raise ValueError("Prefix control shape/dtype differs")
    boundary, first = c["model"]["context"]+1, frame_length(c)
    if not np.array_equal(original[:, :first], control[:, :first]):
        raise ValueError("Initial conditioning frame changed")
    marker = original >= c["nca"]["states"]**4
    if not np.array_equal(marker, control >= c["nca"]["states"]**4) or not np.array_equal(original[marker], control[marker]):
        raise ValueError("Prefix control markers changed")
    for a, b in ((first, boundary), (boundary, original.shape[1])):
        if not np.array_equal(np.sort(original[:, a:b], axis=1), np.sort(control[:, a:b], axis=1)):
            raise ValueError("Supervised-prefix or unused-suffix histogram changed")


In [ ]:
%%writefile transfer_v4/data.py
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import time
import urllib.request

import numpy as np

from .common import dump, sha_file, source_hashes
from .nca import shuffled
from .graph import generate_paired, rewire, topology_hash
from .controls import prefix_shuffled, validate_prefix_control


def download(url, path):
    temporary = Path(str(path)+".part")
    req = urllib.request.Request(url, headers={"User-Agent": "NCA-prepretraining-transfer-v4"})
    try:
        with urllib.request.urlopen(req, timeout=90) as src, temporary.open("wb") as dst:
            while chunk := src.read(1024*1024):
                dst.write(chunk)
        temporary.replace(path)
    except Exception:
        # Preserve failed download for diagnosis; never consume a partial file.
        raise


def prepare_text(c, root, fixture_dir=None):
    started = time.perf_counter()
    provenance = {"source": c["text"], "splits": {}, "tokenizer": c["text"]["tokenizer"]}
    if fixture_dir is not None:
        if c["purpose"] != "execution_check":
            raise ValueError("Synthetic local text fixtures are not research data")
        splits = ["train", "validation"]+(["test"] if c["text"]["evaluate_test"] else [])
        for split in splits:
            raw = (Path(fixture_dir)/f"{split}.txt").read_bytes()
            np.save(root / f"text_{split}.npy", np.frombuffer(raw, dtype=np.uint8).astype(np.uint16))
            provenance["splits"][split] = {"bytes_sha256": hashlib.sha256(raw).hexdigest(), "tokens": len(raw)}
        provenance["vocab"] = 256
        provenance["test_status"] = "execution_fixture_only" if "test" in splits else "not_loaded"
    else:
        if c["text"]["tokenizer"] != "gpt2":
            raise ValueError("Execution checks require an explicit --fixture-dir")
        import pyarrow.parquet as pq
        os.environ.setdefault("TIKTOKEN_CACHE_DIR", str((root/"tokenizer_cache").resolve()))
        import tiktoken
        encoder = tiktoken.get_encoding("gpt2")
        provenance["vocab"] = encoder.n_vocab
        provenance["tokenizer_package_version"] = importlib.metadata.version("tiktoken")
        provenance["parquet_reader_version"] = importlib.metadata.version("pyarrow")
        # Fingerprint the actual merge table + special tokens, not just its name.
        merge = sorted((token.hex(), rank) for token, rank in encoder._mergeable_ranks.items())
        provenance["tokenizer_sha256"] = hashlib.sha256(json.dumps([merge, encoder._special_tokens], separators=(",", ":")).encode()).hexdigest()
        splits = ["train", "validation"]+(["test"] if c["text"]["evaluate_test"] else [])
        for split in splits:
            url = (f"https://huggingface.co/datasets/{c['text']['dataset']}/resolve/{c['text']['revision']}/"
                   f"{c['text']['subset']}/{split}-00000-of-00001.parquet")
            path = root / f"wikitext_{split}.parquet"
            download(url, path)
            rows = pq.read_table(path, columns=["text"])["text"].to_pylist()
            raw = "\n".join(rows)
            ids = np.array(encoder.encode_ordinary(raw), dtype=np.uint16)
            np.save(root / f"text_{split}.npy", ids)
            provenance["splits"][split] = {"url": url, "parquet_sha256": sha_file(path),
                                           "text_sha256": hashlib.sha256(raw.encode()).hexdigest(),
                                           "tokens": len(ids), "rows": len(rows)}
        provenance["test_status"] = "loaded_for_predeclared_final_evaluation" if "test" in splits else "reserved_not_downloaded"
        provenance["license"] = "WikiText: CC-BY-SA-3.0 / GFDL; see Salesforce/wikitext dataset card"
    provenance["seconds"] = time.perf_counter()-started
    return provenance


def prepare(c, root, device, fixture_dir=None):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    import torch
    manifest = {"config": c, "text": prepare_text(c, root, fixture_dir), "replicates": {}, "files": {},
                "generation": {"device": str(device), "torch": str(torch.__version__), "numpy": np.__version__,
                               "source_sha256": source_hashes()}}
    for seed in c["seeds"]:
        folder = root / f"replicate_{seed}"
        folder.mkdir()
        edges, graph_stats = rewire(c["nca"]["grid"], c["graph"]["rewire_fraction"], c["graph"]["topology_seed"] + seed)
        np.save(folder / "graph_edges.npy", edges)
        rep = {"text": manifest["text"], "nca": {}, "graph": graph_stats, "shared_finalize_seconds": 0.}
        for split, count in (("train", c["nca"]["train_rules"]), ("validation", c["nca"]["val_rules"])):
            tick = time.perf_counter()
            try:
                paired, rules, stats = generate_paired(c, count, split, seed, device, edges)
            except Exception as exc:
                dump(root / "PREPARATION_FAILED.json", {"replicate": seed, "stage": split, "error": str(exc), "config": c})
                raise
            for kind, tokens in paired.items():
                np.save(folder / f"{kind}_{split}.npy", tokens)
                shuffled_kind = "shuffled" if kind == "nca" else "shuffled_graph"
                np.save(folder / f"{shuffled_kind}_{split}.npy",
                        shuffled(tokens, c, c["nca"]["data_seed"] + 100003*seed + (10 if split == "train" else 20)))
                prefix_kind = "prefix_grid" if kind == "nca" else "prefix_graph"
                control = prefix_shuffled(tokens, c, c["nca"]["data_seed"] + 100003*seed + (30 if split == "train" else 40))
                validate_prefix_control(tokens, control, c)
                np.save(folder / f"{prefix_kind}_{split}.npy", control)
            np.savez(folder / f"rules_{split}.npz", **rules)
            stats["prepare_seconds"] = time.perf_counter() - tick
            rep["nca"][split] = stats
        if set(rep["nca"]["train"]["rule_ids"]) & set(rep["nca"]["validation"]["rule_ids"]):
            raise RuntimeError("Train/validation rule identities overlap")
        manifest["replicates"][str(seed)] = rep
    for path in sorted(root.rglob("*")):
        if path.suffix in (".npy", ".npz", ".parquet"):
            manifest["files"][path.relative_to(root).as_posix()] = sha_file(path)
    manifest["total_prepare_seconds"] = time.perf_counter()-started
    manifest["prefix_control"] = {"conditioning": "Unchanged first full frame",
                                  "histogram_scope": "Per trajectory, supervised prefix and unused suffix separately",
                                  "boundary_exclusive": c["model"]["context"]+1,
                                  "markers": "Unchanged values and positions",
                                  "selection": "Generated after joint rule selection; controls never affect admission"}
    manifest["shared_finalize_seconds"] = max(0., manifest["total_prepare_seconds"]-manifest["text"]["seconds"]
                                               -sum(s["prepare_seconds"] for r in manifest["replicates"].values() for s in r["nca"].values()))
    for rep in manifest["replicates"].values():
        rep["shared_finalize_seconds"] = manifest["shared_finalize_seconds"]
    dump(root / "manifest.json", manifest)
    return manifest


def load_data(root, c):
    root = Path(root)
    manifest = json.loads((root / "manifest.json").read_text())
    if manifest["config"] != c:
        raise ValueError("Cached data/config mismatch")
    for name, expected in manifest["files"].items():
        if sha_file(root / name) != expected:
            raise ValueError(f"Cached file checksum mismatch: {name}")
    text = {path.stem: np.load(path, allow_pickle=False) for path in root.glob("text_*.npy")}
    arrays = {}
    for seed in c["seeds"]:
        arrays[str(seed)] = dict(text, **{p.stem: np.load(p, allow_pickle=False) for p in (root/f"replicate_{seed}").glob("*.npy")})
        if topology_hash(arrays[str(seed)]["graph_edges"]) != manifest["replicates"][str(seed)]["graph"]["sha256"]:
            raise ValueError("Graph identity mismatch")
    return arrays, manifest


In [ ]:
%%writefile transfer_v4/graph.py
"""Slot-labeled, degree-preserving graph NCA and paired rule selection.

Grid and graph use the SAME message function. Only the donor-node indices change.
The 2x2 tokenizer uses fixed node labels, not an arbitrary graph layout optimizer.
"""
from __future__ import annotations

import hashlib
import math
import time

import numpy as np
import torch

from .common import frame_length, sync
from .nca import complexity, encode_frames, lattice, rollout, sample_rules


def topology_hash(edges):
    return hashlib.sha256(np.asarray(edges, dtype="<i8").tobytes()).hexdigest()


def validate_topology(edges, side):
    edges = np.asarray(edges)
    nodes = side * side
    if edges.shape != (nodes, 9) or not np.issubdtype(edges.dtype, np.integer):
        raise ValueError("Expected nine integer donor slots per node")
    if np.any(edges < 0) or np.any(edges >= nodes) or not np.array_equal(edges[:, 4], np.arange(nodes)):
        raise ValueError("Invalid donor index or center/self slot")
    if any(len(set(row.tolist())) != 9 for row in edges):
        raise ValueError("Duplicate neighbors or extra self edges")
    if any(not np.array_equal(np.sort(edges[:, j]), np.arange(nodes)) for j in range(9)):
        raise ValueError("Each slot must preserve one incoming/outgoing edge per node")


def rewire(side, fraction, seed):
    if side < 4 or not 0 <= fraction <= 1:
        raise ValueError("Invalid graph size/fraction")
    original = lattice(side)
    edges = original.copy()
    rng = np.random.default_rng(seed)
    slots = np.array([0, 1, 2, 3, 5, 6, 7, 8])
    desired = math.ceil(fraction * side * side * 8)
    attempts = swaps = 0
    while np.count_nonzero(edges[:, slots] != original[:, slots]) < desired:
        attempts += 1
        if attempts > 1000 * side * side * 8:
            raise RuntimeError("Cannot reach declared rewiring fraction without violating graph invariants")
        j = int(rng.choice(slots))
        u, v = rng.choice(side * side, 2, replace=False)
        a, b = edges[u, j], edges[v, j]
        if b in edges[u] or a in edges[v]:
            continue
        edges[u, j], edges[v, j] = b, a
        swaps += 1
    validate_topology(edges, side)
    changed = float(np.mean(edges[:, slots] != original[:, slots]))
    return edges, {"seed": seed, "requested_changed_fraction": fraction,
                   "actual_changed_fraction": changed, "accepted_swaps": swaps,
                   "attempts": attempts, "sha256": topology_hash(edges),
                   "grid_sha256": topology_hash(original),
                   "kind": "Directed slot-labeled donor graph; fixed self slot; eight distinct other donors; per-slot degree preserved"}


def paired_rollout(rules, c, frames, generator, device, edges):
    before = generator.get_state()
    grid = rollout(rules, c, frames, generator, device)
    after = generator.get_state()
    generator.set_state(before)
    graph = rollout(rules, c, frames, generator, device, edges=edges)
    if not torch.equal(after, generator.get_state()):
        raise RuntimeError("Paired rollout random draws diverged")
    return grid, graph


@torch.no_grad()
def generate_paired(c, count, split, replicate_seed, device, edges):
    n = c["nca"]
    if n["channel_padding"] != "spatial_only":
        raise ValueError("Spatial-only padding is required")
    seed = n["data_seed"] + 100003 * replicate_seed + (0 if split == "train" else 1000003)
    generator = torch.Generator(device=device).manual_seed(seed)
    frames = math.ceil((c["model"]["context"] + 1) / frame_length(c))
    arrays = {"nca": [], "graph": []}
    profile_scores = {k: [] for k in arrays}
    fresh_scores = {k: [] for k in arrays}
    fixed = {k: [] for k in arrays}
    saved_rules, ids = {}, []
    candidates = accepted = 0
    individual_passes = {k: 0 for k in arrays}
    started = time.perf_counter()
    while accepted < count:
        b = min(n["candidate_batch"], n["max_candidates"] - candidates)
        if b <= 0:
            raise RuntimeError(f"Paired complexity filter exhausted {candidates} candidates at {accepted}/{count}; no relaxation")
        rules = sample_rules(b, n["states"], generator, device)
        profiles = paired_rollout(rules, c, n["profile_frames"], generator, device, edges)
        scores = {k: complexity(encode_frames(x, n["grid"], n["states"]).cpu().numpy(), c)
                  for k, x in zip(arrays, profiles)}
        passes = {k: (v > n["filter_low"]) & (v < n["filter_high"]) for k, v in scores.items()}
        for k in arrays:
            individual_passes[k] += int(passes[k].sum())
        selected = np.flatnonzero(passes["nca"] & passes["graph"])[:count - accepted]
        candidate_start = candidates
        candidates += b
        if len(selected):
            chosen = {k: v[selected.tolist()] for k, v in rules.items()}
            actuals = paired_rollout(chosen, c, frames, generator, device, edges)
            for k, actual in zip(arrays, actuals):
                tokens = encode_frames(actual, n["grid"], n["states"]).cpu().numpy().astype(np.uint16)
                arrays[k].append(tokens)
                profile_scores[k].extend(scores[k][selected].tolist())
                fresh_scores[k].extend(complexity(tokens, c).tolist())
                fixed[k].extend((actual[:, -1] == actual[:, -2]).all(-1).cpu().tolist())
            for k, v in chosen.items():
                saved_rules.setdefault(k, []).append(v.cpu().numpy())
            ids.extend(f"{split}:{seed}:{candidate_start + int(i)}" for i in selected)
            accepted += len(selected)
        if candidates % (n["candidate_batch"] * 20) == 0 or accepted == count:
            print(f"Paired {split}, replicate {replicate_seed}: {accepted}/{count}; candidates={candidates}", flush=True)
    sync(device)
    stats = {"replicate_seed": replicate_seed, "generation_seed": seed, "accepted": accepted,
             "candidates": candidates, "rule_ids": ids, "individual_band_passes": individual_passes,
             "generation_seconds": time.perf_counter() - started,
             "selection": "Same rule retained only if BOTH independent grid/graph profiles pass the same strict band",
             "randomness": "Common initial states and sampling draws within each paired rollout; fresh actual rollout after screening",
             "fresh_rollout_note": "Fresh initial conditions may produce complexity outside the screening band"}
    for k in arrays:
        stats[k] = {"profile_scores": profile_scores[k],
                    "profile_gzip_quartiles": np.quantile(profile_scores[k], [.25, .5, .75]).tolist(),
                    "fresh_rollout_gzip_quartiles": np.quantile(fresh_scores[k], [.25, .5, .75]).tolist(),
                    "final_fixed_transition_fraction": float(np.mean(fixed[k]))}
    return ({k: np.concatenate(v) for k, v in arrays.items()},
            {k: np.concatenate(v) for k, v in saved_rules.items()}, stats)


In [ ]:
%%writefile transfer_v4/model.py
"""Small RoPE/RMSNorm/SwiGLU causal transformer; no pretrained external weights."""
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

from .common import state_sha


class RMSNorm(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))

    def forward(self, x):
        return x*torch.rsqrt(x.float().square().mean(-1, keepdim=True)+1e-6).to(x.dtype)*self.weight


def rope(x):
    # Interleaved real/imaginary coordinates; equivalent rotary formulation.
    t, d = x.shape[-2:]
    frequency = 10000.0**(-torch.arange(0, d, 2, device=x.device).float()/d)
    angle = torch.arange(t, device=x.device).float()[:, None]*frequency[None]
    a, b = x[..., 0::2], x[..., 1::2]
    co, si = angle.cos().to(x.dtype), angle.sin().to(x.dtype)
    return torch.stack((a*co-b*si, a*si+b*co), -1).flatten(-2)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        d, h = config["width"], config["heads"]
        hidden = ((int(8*d/3)+63)//64)*64
        self.heads = h
        self.attention_norm, self.ffn_norm = RMSNorm(d), RMSNorm(d)
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.output = nn.Linear(d, d, bias=False)
        self.gate, self.up, self.down = (nn.Linear(d, hidden, bias=False),
                                       nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False))

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(self.attention_norm(x)).reshape(b, t, 3, self.heads, d//self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(rope(q), rope(k), v, dropout_p=0., is_causal=True)
        x = x+self.output(a.transpose(1, 2).reshape(b, t, d))
        z = self.ffn_norm(x)
        return x+self.down(F.silu(self.gate(z))*self.up(z))


class Backbone(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config["layers"])])
        self.norm = RMSNorm(config["width"])

    def forward(self, x):
        return self.norm(self.blocks(x))


class LM(nn.Module):
    def __init__(self, config, vocab, core_seed, head_seed):
        super().__init__()
        # Separate streams ensure vocabulary-size changes cannot change the
        # paired initial backbone. Construction happens on CPU, then moves.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(core_seed)
            self.core = Backbone(config)
            for module in self.core.modules():
                if isinstance(module, nn.Linear):
                    nn.init.normal_(module.weight, std=.02)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(head_seed)
            self.embedding = nn.Embedding(vocab, config["width"])
            nn.init.normal_(self.embedding.weight, std=.02)
        self.vocab = vocab
        self.context = config["context"]

    def forward(self, tokens):
        if tokens.shape[1] > self.context:
            raise ValueError("Sequence exceeds declared context")
        z = self.core(self.embedding(tokens))
        return F.linear(z, self.embedding.weight)  # Explicitly tied input/output table.

    def core_hash(self):
        return state_sha(self.core.state_dict())

    def head_hash(self):
        return state_sha(self.embedding.state_dict())


def language_model(c, vocab, seed, ppt_state=None):
    model = LM(c["model"], vocab, core_seed=seed, head_seed=seed+100_000)
    if ppt_state is not None:
        # Never transfer synthetic embeddings, optimizer moments, or scheduler.
        model.core.load_state_dict(ppt_state, strict=True)
    return model


In [ ]:
%%writefile transfer_v4/nca.py
"""Categorical spatial NCA port, frozen random rules and rule-disjoint datasets.

Reference: danihyunlee/nca-pre-pretraining at bdd1c71e04eb2e5a2c91481faf278e95846b4431.
This implements spatial-only padding and uniform initial states as described in
the paper, rather than inheriting the reference's channel-padding side effect.
"""
from __future__ import annotations

import gzip
import math
import time

import numpy as np
import torch
from torch.nn import functional as F

from .common import frame_length, sync


def lattice(side):
    ids = np.arange(side*side).reshape(side, side)
    return np.stack([np.roll(ids, (-dy, -dx), (0, 1)).ravel()
                     for dy in (-1, 0, 1) for dx in (-1, 0, 1)], 1)


def sample_rules(count, states, generator, device):
    def lecun(shape, fan_in):
        # Flax-style fan-in variance with a normal truncated to [-2,2].
        u = torch.rand(shape, generator=generator, device=device)*.9544997361+.02275013195
        return (2*u-1).erfinv()*math.sqrt(2/fan_in)/.8796256610
    return {"perception": lecun((count, 9*states, 4), 9*states),
            "hidden": lecun((count, 4, 16), 4),
            "output": lecun((count, 16, states), 16)}


def message_logits(state, edges, rules, states):
    local = F.one_hot(state[:, edges], states).float().flatten(-2)
    perception = torch.einsum("bni,bic->bnc", local, rules["perception"])
    hidden = F.relu(torch.einsum("bnc,bch->bnh", perception, rules["hidden"]))
    return torch.einsum("bnh,bhs->bns", hidden, rules["output"])


def convolution_logits(state, rules, side, states):
    # Independent spatial-convolution formulation for the equivalence test.
    b = state.shape[0]
    x = F.one_hot(state, states).float().reshape(b, side, side, states).permute(0, 3, 1, 2)
    weight = rules["perception"].reshape(b, 9, states, 4).permute(0, 3, 2, 1).reshape(b*4, states, 3, 3)
    perceived = F.conv2d(F.pad(x.reshape(1, b*states, side, side), (1, 1, 1, 1), mode="circular"), weight, groups=b)
    p = perceived.reshape(b, 4, side*side).transpose(1, 2)
    hidden = F.relu(torch.einsum("bnc,bch->bnh", p, rules["hidden"]))
    return torch.einsum("bnh,bhs->bns", hidden, rules["output"])


def step(state, edges, rules, states, temperature, generator):
    logits = message_logits(state, edges, rules, states)
    if temperature > 0:
        u = torch.rand(logits.shape, device=logits.device, generator=generator).clamp(1e-7, 1-1e-7)
        logits = logits-temperature*torch.log(-torch.log(u))
    return logits.argmax(-1)


def rollout(rules, c, frames, generator, device, edges=None):
    n = c["nca"]
    edges = torch.as_tensor(lattice(n["grid"]) if edges is None else edges, dtype=torch.long, device=device)
    batch = rules["perception"].shape[0]
    state = torch.randint(n["states"], (batch, n["grid"]**2), generator=generator, device=device)
    for _ in range(n["burnin"]):
        state = step(state, edges, rules, n["states"], n["temperature"], generator)
    values = []
    for _ in range(frames):
        values.append(state)
        state = step(state, edges, rules, n["states"], n["temperature"], generator)
    return torch.stack(values, 1)


def encode_frames(frames, side, states):
    x = frames.reshape(*frames.shape[:-1], side, side)
    patches = (x[..., 0::2, 0::2]+states*x[..., 0::2, 1::2]
               +states**2*x[..., 1::2, 0::2]+states**3*x[..., 1::2, 1::2]).flatten(-2)
    start = torch.full((*patches.shape[:-1], 1), states**4, dtype=torch.long, device=frames.device)
    return torch.cat((start, patches, start+1), -1).flatten(1)


def complexity(tokens, c):
    patches = tokens.reshape(tokens.shape[0], -1, frame_length(c))[:, :, 1:-1]
    scores = []
    for row in patches:
        raw = np.asarray(row, dtype="<i4").tobytes(order="C")
        scores.append(len(gzip.compress(raw, compresslevel=9, mtime=0))/len(raw))
    return np.array(scores)



def shuffled(tokens, c, seed):
    result = tokens.copy()
    rng = np.random.default_rng(seed)
    cutoff = c["nca"]["states"]**4
    for row in result:
        mask = row < cutoff
        row[mask] = rng.permutation(row[mask])
    return result


def targets(tokens, c, patch_only=False):
    labels = tokens[:, 1:].clone()
    absolute = torch.arange(1, tokens.shape[1], device=tokens.device)
    mask = absolute < frame_length(c)
    labels[:, mask] = -100
    if patch_only:
        labels[labels >= c["nca"]["states"]**4] = -100
    return labels


def control_probabilities(train, c):
    cutoff, length = c["nca"]["states"]**4, c["model"]["context"]
    # Fit controls on the SAME trained prefix, never on held-out rules.
    prefix = train[:, :length+1].astype(np.int64)
    y = prefix[:, 1:]
    absolute = np.arange(1, length+1)
    mask = (absolute[None] >= frame_length(c)) & (y < cutoff)
    counts = np.bincount(y[mask], minlength=cutoff).astype(np.float64)+1
    probabilities = counts/counts.sum()
    previous = prefix[:, np.maximum(0, absolute-frame_length(c))]
    truth, before = y[mask], previous[mask]
    base = probabilities[truth]
    copy = (truth == before).astype(np.float64)
    mixture_grid = np.linspace(0, .99, 100)
    losses = [-np.log((1-q)*base+q*copy).mean() for q in mixture_grid]
    mixture = float(mixture_grid[int(np.argmin(losses))])
    return probabilities, mixture


def evaluate_controls(tokens, c, probabilities, mixture):
    length, f = c["model"]["context"], frame_length(c)
    prefix = tokens[:, :length+1].astype(np.int64)
    y = prefix[:, 1:]
    absolute = np.arange(1, length+1)
    mask = (absolute[None] >= f) & (y < len(probabilities))
    previous = prefix[:, np.maximum(0, absolute-f)]
    truth, before = y[mask], previous[mask]
    base = probabilities[truth]
    mixture_probability = (1-mixture)*base+mixture*(truth == before)
    return {"unigram_patch_nll": float(-np.log(base).mean()),
            "copy_mixture_patch_nll": float(-np.log(mixture_probability).mean()),
            "copy_mixture_weight_fitted_on_train": mixture,
            "unigram_patch_accuracy": float(np.mean(truth == probabilities.argmax())),
            "previous_frame_patch_accuracy": float(np.mean(truth == before)),
            "patch_targets": int(mask.sum())}


In [ ]:
%%writefile transfer_v4/plan.py
"""Archive a prospective analysis plan before any benchmark model is trained."""
from pathlib import Path
import json

from .common import STUDY, dump, sha_file, source_hashes


def record_plan(root, c):
    root = Path(root)
    protocol = Path(__file__).resolve().parents[1]/"docs/TRANSFER_V4_PROTOCOL.md"
    (root/"FROZEN_PROTOCOL.md").write_bytes(protocol.read_bytes())
    dump(root/"analysis_plan.json", {"config": c, "study": STUDY,
                                   "source_sha256": source_hashes(),
                                   "protocol_sha256": sha_file(root/"FROZEN_PROTOCOL.md"),
                                   "scope": "Prospective validation replication and mechanism study, not untouched-domain confirmation"})
    return sha_file(root/"analysis_plan.json")


def verify_plan(root, c):
    root = Path(root)
    env = json.loads((root/"environment.json").read_text())
    if sha_file(root/"analysis_plan.json") != env["analysis_plan_sha256"]:
        raise ValueError("Frozen analysis plan checksum mismatch")
    plan = json.loads((root/"analysis_plan.json").read_text())
    if plan["config"] != c or plan["study"] != STUDY or plan["source_sha256"] != source_hashes():
        raise ValueError("Frozen analysis plan identity mismatch")
    if sha_file(root/"FROZEN_PROTOCOL.md") != plan["protocol_sha256"]:
        raise ValueError("Frozen protocol checksum mismatch")


In [ ]:
%%writefile transfer_v4/report.py
from __future__ import annotations

import csv
import json
import math
from pathlib import Path
import statistics

import numpy as np

from .common import ARMS, SYNTHETIC_ARMS, dump
from .analysis import common_target_analysis


def first_hit(curve, threshold):
    return next((r for r in curve if r["validation_nll"] <= threshold), None)


def report(root, c):
    root = Path(root)
    status = json.loads((root/"status.json").read_text())
    summaries, curves = {}, {}
    for arm in ARMS:
        for seed in c["seeds"]:
            folder = root/f"{arm}_{seed}"
            if (folder/"language_summary.json").exists():
                summaries[arm, seed] = json.loads((folder/"language_summary.json").read_text())
                curves[arm, seed] = [json.loads(s) for s in (folder/"language_metrics.jsonl").read_text().splitlines()]
    comparisons = []
    for seed in c["seeds"]:
        if ("scratch", seed) not in summaries:
            continue
        reference = summaries["scratch", seed]["final_validation_nll"]
        base_hit = first_hit(curves["scratch", seed], reference)
        for arm in SYNTHETIC_ARMS:
            if (arm, seed) not in summaries:
                continue
            hit = first_hit(curves[arm, seed], reference)
            row = {"arm": arm, "seed": seed, "reference_nll": reference,
                   "reference_policy": "Paired scratch final validation loss at its declared N language updates",
                   "reference_reached": hit is not None,
                   "language_step_at_reference": hit["language_step"] if hit else None,
                   "final_validation_nll_delta": summaries[arm, seed]["final_validation_nll"]-reference}
            for key in ("language_tokens", "language_train_seconds", "language_seconds", "total_training_tokens", "cumulative_seconds"):
                row[f"{key}_speedup"] = base_hit[key]/hit[key] if hit and hit[key] > 0 else None
            comparisons.append(row)
    aggregate = []
    for arm in SYNTHETIC_ARMS:
        rows = [r for r in comparisons if r["arm"] == arm]
        if not rows:
            continue
        deltas = [r["final_validation_nll_delta"] for r in rows]
        a = {"arm": arm, "paired_seeds": len(rows), "seeds_reaching_reference": sum(r["reference_reached"] for r in rows),
             "mean_final_validation_nll_delta": statistics.mean(deltas)}
        for key in ("language_tokens", "language_seconds", "cumulative_seconds"):
            values = [r[f"{key}_speedup"] for r in rows]
            # Do not publish a success-only mean when any seed is censored.
            complete = all(v is not None and v > 0 for v in values)
            a[f"geomean_{key}_speedup_all_seeds"] = math.exp(statistics.mean(math.log(v) for v in values)) if complete else None
            if complete and len(values) >= 2:
                rng = np.random.default_rng(101)
                samples = np.exp(np.mean(rng.choice(np.log(values), (10000, len(values)), replace=True), axis=1))
                a[f"exploratory_95_ci_{key}"] = np.quantile(samples, [.025, .975]).tolist()
            else:
                a[f"exploratory_95_ci_{key}"] = None
        aggregate.append(a)
    treatment_comparisons = []
    for reference_arm, arm in (("grid_nca", "graph_nca"), ("shuffled_graph", "graph_nca"), ("shuffled_nca", "grid_nca")):
        rows = []
        for seed in c["seeds"]:
            if (reference_arm, seed) not in summaries or (arm, seed) not in summaries:
                continue
            threshold = summaries[reference_arm, seed]["final_validation_nll"]
            base = first_hit(curves[reference_arm, seed], threshold)
            hit = first_hit(curves[arm, seed], threshold)
            row = {"arm": arm, "reference_arm": reference_arm, "seed": seed,
                   "reference_nll": threshold, "reference_reached": hit is not None,
                   "language_step_at_reference": hit["language_step"] if hit else None,
                   "final_validation_nll_delta": summaries[arm, seed]["final_validation_nll"] - threshold}
            for key in ("language_tokens", "language_seconds", "cumulative_seconds"):
                row[key+"_speedup"] = base[key]/hit[key] if hit and hit[key] > 0 else None
            rows.append(row)
        if rows:
            item = {"arm": arm, "reference_arm": reference_arm, "paired_replicates": len(rows),
                    "seeds_reaching_reference": sum(r["reference_reached"] for r in rows),
                    "mean_final_validation_nll_delta": statistics.mean(r["final_validation_nll_delta"] for r in rows),
                    "comparisons": rows}
            for key in ("language_tokens", "language_seconds", "cumulative_seconds"):
                values = [r[key+"_speedup"] for r in rows]
                item["geomean_"+key+"_speedup_all_seeds"] = math.exp(statistics.mean(math.log(v) for v in values)) if all(v is not None and v > 0 for v in values) else None
            treatment_comparisons.append(item)
    result = {"schema_version": 4, "purpose": c["purpose"], "status": status,
              "primary_question": "Does graph PPT beat grid PPT and exact-prefix shuffled graph PPT at the same paired scratch-endpoint target?",
              "primary_analysis": common_target_analysis(curves, c),
              "secondary_question": "Do language-stage savings repay PPT and generation costs?",
              "graph_status": "Paired grid/graph frozen rules, same complexity band and independent topology/cache per paired model seed",
              "treatment_comparisons": treatment_comparisons,
              "comparisons": comparisons, "aggregate": aggregate,
              "uncertainty_note": "Six fresh paired model/data/topology cohorts do not separately estimate these variance sources. Validation task was inspected in earlier versions. Evaluation crossings are discrete; a gate pass is not proof of dynamics-specific transfer. Legacy reference-endpoint comparisons below are secondary.",
              "test_policy": "Reserved and not loaded" if not c["text"]["evaluate_test"] else "Predeclared final evaluation only"}
    dump(root/"report.json", result)
    plot_ppt(root, c)
    if summaries:
        flat = [{k: v for k, v in s.items() if not isinstance(v, dict)} for s in summaries.values()]
        with (root/"summary.csv").open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(flat[0]))
            w.writeheader()
            w.writerows(flat)
        plot(root, c, curves)
    print(json.dumps({"status": status, "primary_analysis": result["primary_analysis"], "aggregate": aggregate}, indent=2), flush=True)
    return result


def plot_ppt(root, c):
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for arm, color in (("grid_nca", "#2563eb"), ("graph_nca", "#059669"),
                       ("shuffled_nca", "#db2777"), ("shuffled_graph", "#d97706"),
                       ("prefix_grid", "#9333ea"), ("prefix_graph", "#0891b2")):
        for i, seed in enumerate(c["seeds"]):
            folder = Path(root)/f"{arm}_{seed}"
            if not (folder/"ppt_summary.json").exists():
                continue
            rows = [json.loads(s) for s in (folder/"ppt_metrics.jsonl").read_text().splitlines()]
            summary = json.loads((folder/"ppt_summary.json").read_text())
            label = arm if i == 0 else None
            trained = [r for r in rows if r["train_loss"] is not None]
            axes[0].plot([r["ppt_step"] for r in trained], [r["train_loss"] for r in trained], color=color, alpha=.65, label=label)
            axes[1].plot([r["ppt_step"] for r in rows], [r["validation"]["patch_nll"] for r in rows], color=color, alpha=.65)
            axes[2].plot([r["ppt_step"] for r in rows], [r["validation"]["patch_accuracy"] for r in rows], color=color, alpha=.65)
            if i == 0:
                bound = min(summary["controls"]["unigram_patch_nll"], summary["controls"]["copy_mixture_patch_nll"])
                axes[1].axhline(bound, color=color, linestyle="--", linewidth=.8)
    for ax, title, ylabel in zip(axes, ("Training objective", "Unseen-rule patch prediction", "Unseen-rule accuracy"),
                                 ("Cross-entropy (conditioning masked)", "Patch NLL (dashed: best fitted control)", "Patch accuracy")):
        ax.set_title(title)
        ax.set_xlabel("PPT updates")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    fig.suptitle("Synthetic-stage diagnostics — a gate pass does not establish language transfer")
    fig.savefig(Path(root)/"ppt_diagnostics.png", dpi=180)
    fig.savefig(Path(root)/"ppt_diagnostics.svg")
    plt.close(fig)


def plot(root, c, curves):
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot as plt
    colors = {"scratch": "#64748b", "grid_nca": "#2563eb", "graph_nca": "#059669",
              "shuffled_nca": "#db2777", "shuffled_graph": "#d97706",
              "prefix_grid": "#9333ea", "prefix_graph": "#0891b2"}
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    keys = ("language_tokens", "language_seconds", "cumulative_seconds")
    for (arm, seed), rows in curves.items():
        for ax, key in zip(axes, keys):
            scale = 1e6 if key == "language_tokens" else 1
            ax.plot([r[key]/scale for r in rows], [r["validation_nll"] for r in rows],
                    color=colors[arm], alpha=.6, label=arm if seed == c["seeds"][0] else None)
    for ax, title, xlabel in zip(axes, ("Language exposure", "Language-stage time", "PPT + language cost"),
                                 ("Language tokens (millions)", "Seconds since language step 0", "Recorded cumulative seconds")):
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Validation NLL (nats per language token)")
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    prefix = "Execution fixture; no research inference" if c["purpose"] == "execution_check" else "Graph vs grid pre-pre-training into identical language stages"
    fig.suptitle(prefix)
    fig.savefig(root/"learning_curves.png", dpi=180)
    fig.savefig(root/"learning_curves.svg")
    plt.close(fig)


In [ ]:
%%writefile transfer_v4/run.py
"""CLI: python -m transfer_v4.run all --config configs/transfer_v4.json ..."""
from __future__ import annotations

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import argparse
from datetime import datetime, timezone
import json
from pathlib import Path
import platform
import sys
import time

import numpy as np
import torch

from .common import ARMS, SYNTHETIC_ARMS, REFERENCE_COMMIT, dump, load_config, sha_file, source_hashes, sync
from .model import language_model
from .calibrate import calibrate
from .data import load_data, prepare
from .report import report
from .training import train_language, train_ppt
from .verify import verify
from .plan import record_plan, verify_plan


def setup(device):
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but no CUDA GPU is available")


def initialize_run(c, root, data_root, device):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=False)
    plan_hash = record_plan(root, c)
    text = json.loads((Path(data_root)/"manifest.json").read_text())["text"]
    initializations = {}
    for seed in c["seeds"]:
        model = language_model(c, text["vocab"], seed)
        path = root/f"initialization_{seed}.pt"
        torch.save({"schema_version": 4, "config": c, "seed": seed,
                    "core": model.core.state_dict(), "embedding": model.embedding.state_dict()}, path)
        initializations[str(seed)] = {"file": path.name, "sha256": sha_file(path),
                                      "core_sha256": model.core_hash(), "head_sha256": model.head_hash()}
    del model
    dump(root/"environment.json", {
        "schema_version": 4, "config": c, "started_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version, "torch": str(torch.__version__), "platform": platform.platform(),
        "numpy": np.__version__,
        "device": device, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "cuda": torch.version.cuda, "tf32": False, "reference_commit": REFERENCE_COMMIT,
        "source_sha256": source_hashes(), "data_manifest_sha256": sha_file(Path(data_root)/"manifest.json"),
        "initializations": initializations,
        "analysis_plan_sha256": plan_hash,
        "graph_treatment": "Paired frozen rules, slot-labeled degree-preserving rewiring, independent cache/topology per model seed",
        "timing_scope": "Language seconds start after model/head/optimizer construction and include validation. Cumulative charges text preparation, the entire paired synthetic preparation/cache writes per replicate for every synthetic arm, language setup, shared manifest work and PPT. Excludes initial common-weight snapshot creation, package startup, cloud queue, disposable calibration and final language checkpoint/verification/plot/archive work."})
    dump(root/"status.json", {"state": "ppt_pending", "purpose": c["purpose"]})


def check_identity(root, c, data_root):
    verify_plan(root, c)
    env = json.loads((Path(root)/"environment.json").read_text())
    if env["config"] != c or env["data_manifest_sha256"] != sha_file(Path(data_root)/"manifest.json"):
        raise ValueError("Run/config/data mismatch")
    if env["source_sha256"] != source_hashes():
        raise ValueError("Source changed during this staged run; create a fresh run")


def ppt_stage(c, root, arrays, manifest_hash, device):
    summaries = []
    dump(Path(root)/"status.json", {"state": "ppt_running", "purpose": c["purpose"]})
    for i, seed in enumerate(c["seeds"]):
        order = SYNTHETIC_ARMS[i % len(SYNTHETIC_ARMS):]+SYNTHETIC_ARMS[:i % len(SYNTHETIC_ARMS)]
        for arm in order:
            summaries.append(train_ppt(c, arm, seed, arrays[str(seed)], manifest_hash, root, device))
    failed = [{"arm": s["arm"], "seed": s["seed"]} for s in summaries
              if s["arm"] in ("grid_nca", "graph_nca") and not s["gate"]["passed"]]
    allowed = not failed or (c["purpose"] == "execution_check" and not c["gate"]["enforce"])
    status = {"state": "ppt_ready" if allowed else "stopped_at_synthetic_gate", "purpose": c["purpose"],
              "structured_seeds_failing_gate": failed, "language_allowed": allowed,
              "execution_only_bypass": bool(failed and allowed),
              "note": "Passing this gate only permits testing transfer; it does not establish acceleration."}
    dump(Path(root)/"status.json", status)
    dump(Path(root)/"ppt_gate.json", {"status": status, "summaries": summaries})
    return allowed


def language_stage(c, root, arrays, manifest, manifest_hash, device):
    status = json.loads((Path(root)/"status.json").read_text())
    if status["state"] != "ppt_ready" or not status["language_allowed"]:
        raise ValueError("Held-out synthetic diagnostics must permit language training first")
    dump(Path(root)/"status.json", {"state": "language_running", "purpose": c["purpose"]})
    for i, seed in enumerate(c["seeds"]):
        order = ARMS[i % len(ARMS):]+ARMS[:i % len(ARMS)]
        summaries = []
        for arm in order:
            summaries.append(train_language(c, arm, seed, arrays[str(seed)], manifest["replicates"][str(seed)], manifest_hash, root, device))
        for key in ("core_initial_sha256", "language_head_initial_sha256", "language_train_order_sha256", "language_schedule_sha256"):
            if len({s[key] for s in summaries}) != 1:
                raise RuntimeError(f"Language controls failed: {key}")
    dump(Path(root)/"status.json", {"state": "complete", "purpose": c["purpose"],
                                  "scientific_claim": "Inspect paired language-stage curves and censored reference crossings; completion is not success."})


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("command", choices=("prepare", "calibrate", "ppt", "language", "report", "verify", "all"))
    p.add_argument("--config", default="configs/transfer_v4.json")
    p.add_argument("--data", default="data/transfer_v4")
    p.add_argument("--out", default="runs/transfer_v4")
    p.add_argument("--device", choices=("cpu", "cuda", "auto"), default="auto")
    p.add_argument("--fixture-dir", help="Execution checks only: train.txt and validation.txt")
    a = p.parse_args()
    c = load_config(a.config)
    device = "cuda" if a.device == "auto" and torch.cuda.is_available() else "cpu" if a.device == "auto" else a.device
    setup(device)
    if a.command in ("prepare", "all"):
        prepare(c, a.data, device, a.fixture_dir)
    if a.command == "all":
        arrays, manifest = load_data(a.data, c)
        first = str(c["seeds"][0])
        calibrate(c, arrays[first], manifest["replicates"][first], device, a.data)
    if a.command == "calibrate":
        arrays, manifest = load_data(a.data, c)
        first = str(c["seeds"][0])
        calibrate(c, arrays[first], manifest["replicates"][first], device, a.data)
        return
    if a.command in ("ppt", "all"):
        arrays, manifest = load_data(a.data, c)
        initialize_run(c, a.out, a.data, device)
        allowed = ppt_stage(c, a.out, arrays, sha_file(Path(a.data)/"manifest.json"), device)
        if not allowed:
            report(a.out, c)
            verify(a.out, a.data, device)
            print("STOPPED AT SYNTHETIC GATE: diagnostics saved; language benchmark not launched.", flush=True)
            return
    if a.command in ("language", "all"):
        check_identity(a.out, c, a.data)
        arrays, manifest = load_data(a.data, c)
        language_stage(c, a.out, arrays, manifest, sha_file(Path(a.data)/"manifest.json"), device)
    if a.command in ("language", "report", "all"):
        report(a.out, c)
    if a.command in ("language", "verify", "all"):
        verify(a.out, a.data, device)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile transfer_v4/training.py
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import time

import numpy as np
import torch
from torch.nn import functional as F

from .common import DATA_KEYS, dump, effective_tokens, lr_at, seed_all, sha_file, sync
from .model import LM, language_model
from .nca import control_probabilities, evaluate_controls, targets


def append(path, row):
    with Path(path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, allow_nan=False)+"\n")


def batch(array, c, rng, device, synthetic):
    length, size = c["model"]["context"], c["batch"]
    if synthetic:
        indices = rng.integers(0, len(array), size=size)
        values = array[indices, :length+1]
    else:
        if len(array) <= length:
            raise ValueError("Language training split is shorter than context")
        indices = rng.integers(0, len(array)-length, size=size)
        values = array[indices[:, None]+np.arange(length+1)[None]]
    tensor = torch.as_tensor(values.astype(np.int64), device=device)
    labels = targets(tensor, c) if synthetic else tensor[:, 1:]
    return tensor[:, :-1], labels, indices


def eval_rows(array, c, synthetic):
    length = c["model"]["context"]
    if synthetic:
        return array[:min(len(array), c["ppt_eval_rules"]), :length+1]
    count = min(c["language_eval_tokens"]//length, (len(array)-1)//length)
    if count < 1:
        raise ValueError("Evaluation split too short")
    # Target tokens do not overlap across rows. Adjacent rows share only one
    # conditioning boundary token, as in the historical non-overlap evaluation.
    return np.stack([array[i*length:i*length+length+1] for i in range(count)])


@torch.no_grad()
def evaluate(model, array, c, device, synthetic):
    model.eval()
    rows = eval_rows(array, c, synthetic)
    total_loss = patch_loss = 0.
    count = patch_count = correct = 0
    for start in range(0, len(rows), c["eval_batch"]):
        z = torch.as_tensor(rows[start:start+c["eval_batch"]].astype(np.int64), device=device)
        labels = targets(z, c) if synthetic else z[:, 1:]
        logits = model(z[:, :-1])
        losses = F.cross_entropy(logits.transpose(1, 2), labels, reduction="none", ignore_index=-100)
        mask = labels != -100
        total_loss += float(losses[mask].sum())
        count += int(mask.sum())
        if synthetic:
            patch_mask = mask & (labels < c["nca"]["states"]**4)
            patch_loss += float(losses[patch_mask].sum())
            patch_count += int(patch_mask.sum())
            correct += int(((logits.argmax(-1) == labels) & patch_mask).sum())
    model.train()
    result = {"nll": total_loss/count, "targets": count}
    if synthetic:
        result.update(patch_nll=patch_loss/patch_count, patch_accuracy=correct/patch_count,
                      patch_targets=patch_count, rules=len(rows))
    return result


def optimizer_for(model, stage, synthetic):
    if synthetic:
        return torch.optim.Adam(model.parameters(), lr=stage["lr"], weight_decay=stage["weight_decay"])
    return torch.optim.AdamW(model.parameters(), lr=stage["lr"], weight_decay=stage["weight_decay"])


def update(model, optimizer, array, c, stage, local_step, rng, device, synthetic, order_hash):
    lr = lr_at(local_step, stage)
    for group in optimizer.param_groups:
        group["lr"] = lr
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.
    prediction_targets = 0
    for _ in range(c["accumulate"]):
        x, y, indices = batch(array, c, rng, device, synthetic)
        order_hash.update(indices.astype("<i8").tobytes())
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, model.vocab), y.reshape(-1), ignore_index=-100)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite loss")
        (loss/c["accumulate"]).backward()
        loss_sum += float(loss.detach())/c["accumulate"]
        prediction_targets += int((y != -100).sum())
    if stage["clip"] is not None:
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), stage["clip"], error_if_nonfinite=True)
        if not torch.isfinite(norm):
            raise FloatingPointError("Non-finite gradient norm")
    else:
        # A finite global-norm check without clipping; avoids synchronizing the
        # GPU separately for every parameter tensor.
        torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf"), error_if_nonfinite=True)
    optimizer.step()
    return loss_sum, prediction_targets, lr


def quality_gate(initial, selected, controls, c):
    relative = (initial["patch_nll"]-selected["patch_nll"])/initial["patch_nll"]
    baseline = min(controls["unigram_patch_nll"], controls["copy_mixture_patch_nll"])
    margin = baseline-selected["patch_nll"]
    passed = relative >= c["gate"]["minimum_relative_improvement"] and margin >= c["gate"]["baseline_margin_nats"]
    return {"passed": bool(passed), "enforced": c["gate"]["enforce"],
            "relative_patch_nll_improvement": relative, "improvement_over_best_control_nats": margin,
            "required_relative_improvement": c["gate"]["minimum_relative_improvement"],
            "required_baseline_margin_nats": c["gate"]["baseline_margin_nats"],
            "control_scope": "Unseen-rule validation; control probabilities fitted only on synthetic train prefixes"}


def train_ppt(c, arm, seed, arrays, manifest_hash, root, device):
    if arm not in DATA_KEYS:
        raise ValueError("Scratch has no PPT stage")
    sync(device)
    started = time.perf_counter()
    folder = Path(root)/f"{arm}_{seed}"
    folder.mkdir(parents=True, exist_ok=False)
    seed_all(seed)
    kind = DATA_KEYS[arm]
    train, validation = arrays[f"{kind}_train"], arrays[f"{kind}_validation"]
    model = LM(c["model"], c["nca"]["states"]**4+2, core_seed=seed, head_seed=seed+50_000).to(device)
    initialization = torch.load(Path(root)/f"initialization_{seed}.pt", map_location=device, weights_only=True)
    model.core.load_state_dict(initialization["core"], strict=True)
    initial_core = model.core_hash()
    stage = c["ppt"]
    optimizer = optimizer_for(model, stage, synthetic=True)
    rng = np.random.default_rng(seed+101)
    order_hash = hashlib.sha256()
    probabilities, mixture = control_probabilities(train, c)
    selected_rows = eval_rows(validation, c, True)
    controls = evaluate_controls(selected_rows, c, probabilities, mixture)
    initial = evaluate(model, validation, c, device, True)
    best, best_step = initial, 0
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    train_seconds = 0.
    predicted = 0
    window_loss = 0.
    window_count = 0
    append(folder/"ppt_metrics.jsonl", {"ppt_step": 0, "train_loss": None, "validation": initial,
                                       "train_seconds": 0., "stage_seconds": time.perf_counter()-started})
    for k in range(stage["steps"]):
        sync(device)
        tick = time.perf_counter()
        loss, count, lr = update(model, optimizer, train, c, stage, k, rng, device, True, order_hash)
        sync(device)
        train_seconds += time.perf_counter()-tick
        window_loss += loss
        window_count += 1
        predicted += count
        if (k+1) % c["eval_every"] == 0 or k+1 == stage["steps"]:
            measured = evaluate(model, validation, c, device, True)
            sync(device)
            if measured["patch_nll"] < best["patch_nll"]:
                best, best_step = measured, k+1
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            row = {"ppt_step": k+1, "ppt_tokens": (k+1)*effective_tokens(c),
                   "train_loss": window_loss/window_count, "lr": lr, "validation": measured,
                   "train_seconds": train_seconds, "stage_seconds": time.perf_counter()-started}
            append(folder/"ppt_metrics.jsonl", row)
            print(f"PPT {arm} seed={seed} {k+1}/{stage['steps']}: held-out patch NLL={measured['patch_nll']:.4f}, best control={min(controls['unigram_patch_nll'], controls['copy_mixture_patch_nll']):.4f}", flush=True)
            window_loss = 0.
            window_count = 0
    checkpoint = {"schema_version": 4, "config": c, "stage": "ppt", "arm": arm, "seed": seed,
                  "vocab": model.vocab, "data_manifest_sha256": manifest_hash,
                  "core_initial_sha256": initial_core, "selected_step": best_step,
                  "model": best_state, "validation": best}
    torch.save(checkpoint, folder/"ppt_selected.pt")
    # Preserve both selected and final states if validation selection differs.
    if best_step != stage["steps"]:
        final_checkpoint = dict(checkpoint, model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                                selected_step=stage["steps"], validation=measured)
        torch.save(final_checkpoint, folder/"ppt_final.pt")
    gate = quality_gate(initial, best, controls, c)
    summary = {"arm": arm, "seed": seed, "ppt_updates": stage["steps"],
               "ppt_tokens": stage["steps"]*effective_tokens(c), "ppt_prediction_targets": predicted,
               "initial": initial, "selected": best, "selected_step": best_step,
               "controls": controls, "gate": gate, "core_initial_sha256": initial_core,
               "train_order_sha256": order_hash.hexdigest(), "train_seconds": train_seconds,
               "stage_seconds": time.perf_counter()-started,
               "checkpoint": "ppt_selected.pt", "checkpoint_sha256": sha_file(folder/"ppt_selected.pt"),
               "final_checkpoint_sha256": sha_file(folder/"ppt_final.pt") if (folder/"ppt_final.pt").exists() else sha_file(folder/"ppt_selected.pt"),
               "selection_policy": "Minimum held-out-rule patch NLL; full declared PPT budget is charged regardless of selected step"}
    dump(folder/"ppt_summary.json", summary)
    return summary


def load_ppt_core(folder, c, arm, seed, manifest_hash):
    folder = Path(folder)
    summary = json.loads((folder/"ppt_summary.json").read_text())
    path = folder/summary["checkpoint"]
    if sha_file(path) != summary["checkpoint_sha256"]:
        raise ValueError("PPT checkpoint checksum mismatch")
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if saved["config"] != c or saved["arm"] != arm or saved["seed"] != seed or saved["data_manifest_sha256"] != manifest_hash:
        raise ValueError("PPT checkpoint identity mismatch")
    core = {k[len("core."):]: v for k, v in saved["model"].items() if k.startswith("core.")}
    return core, summary


def train_language(c, arm, seed, arrays, manifest, manifest_hash, root, device):
    sync(device)
    setup_started = time.perf_counter()
    folder = Path(root)/f"{arm}_{seed}"
    folder.mkdir(parents=True, exist_ok=True)
    if (folder/"language_metrics.jsonl").exists():
        raise FileExistsError("Refusing to mix language runs; use a fresh output directory")
    seed_all(seed)
    ppt_state, ppt = (None, None) if arm == "scratch" else load_ppt_core(folder, c, arm, seed, manifest_hash)
    model = language_model(c, manifest["text"]["vocab"], seed, ppt_state).to(device)
    initialization = torch.load(Path(root)/f"initialization_{seed}.pt", map_location=device, weights_only=True)
    model.embedding.load_state_dict(initialization["embedding"], strict=True)
    if ppt_state is None:
        model.core.load_state_dict(initialization["core"], strict=True)
    start_core, start_head = model.core_hash(), model.head_hash()
    stage = c["language"]
    optimizer = optimizer_for(model, stage, synthetic=False)
    if optimizer.state:
        raise RuntimeError("Language optimizer must start empty")
    rng = np.random.default_rng(seed+202)
    order_hash, schedule_hash = hashlib.sha256(), hashlib.sha256()
    train_seconds = 0.
    # Joint candidate screening/cache preparation is conservatively charged in
    # full to each synthetic arm. It is not an isolated generator speed metric.
    nca_prepare = sum(v["prepare_seconds"] for v in manifest["nca"].values()) if ppt else 0.
    setup_cost = manifest["text"]["seconds"]+nca_prepare+manifest.get("shared_finalize_seconds", 0.)
    ppt_cost = ppt["stage_seconds"] if ppt else 0.
    sync(device)
    setup_cost += time.perf_counter()-setup_started
    started = time.perf_counter()

    def record(step, train_loss, lr):
        value = evaluate(model, arrays["text_validation"], c, device, False)
        sync(device)
        elapsed = time.perf_counter()-started
        row = {"arm": arm, "seed": seed, "language_step": step,
               "language_tokens": step*effective_tokens(c),
               "ppt_tokens": ppt["ppt_tokens"] if ppt else 0,
               "total_training_tokens": step*effective_tokens(c)+(ppt["ppt_tokens"] if ppt else 0),
               "train_loss": train_loss, "lr": lr, "validation_nll": value["nll"],
               "validation_targets": value["targets"], "language_train_seconds": train_seconds,
               "language_seconds": elapsed, "cumulative_seconds": setup_cost+ppt_cost+elapsed}
        append(folder/"language_metrics.jsonl", row)
        print(f"LANGUAGE {arm} seed={seed} {step}/{stage['steps']}: validation NLL={value['nll']:.4f}", flush=True)
        return row

    first = record(0, None, None)
    window_loss = 0.
    window_count = 0
    for k in range(stage["steps"]):
        sync(device)
        tick = time.perf_counter()
        loss, _, lr = update(model, optimizer, arrays["text_train"], c, stage, k, rng, device, False, order_hash)
        sync(device)
        train_seconds += time.perf_counter()-tick
        schedule_hash.update(np.array([lr], dtype="<f8").tobytes())
        window_loss += loss
        window_count += 1
        if (k+1) % c["eval_every"] == 0 or k+1 == stage["steps"]:
            last = record(k+1, window_loss/window_count, lr)
            window_loss = 0.
            window_count = 0
    # Final validation is never selected using a test set. Confirmation data are
    # neither downloaded nor evaluated unless enabled in the pre-run config.
    test = evaluate(model, arrays["text_test"], c, device, False) if c["text"]["evaluate_test"] else None
    checkpoint = {"schema_version": 4, "config": c, "stage": "language", "arm": arm, "seed": seed,
                  "vocab": model.vocab, "data_manifest_sha256": manifest_hash,
                  "model": model.state_dict(), "language_step": stage["steps"],
                  "validation": {"nll": last["validation_nll"], "targets": last["validation_targets"]},
                  "test": test}
    torch.save(checkpoint, folder/"language_final.pt")
    summary = {"arm": arm, "seed": seed, "purpose": c["purpose"],
               "language_updates": stage["steps"], "language_tokens": stage["steps"]*effective_tokens(c),
               "ppt_updates": c["ppt"]["steps"] if ppt else 0, "ppt_tokens": ppt["ppt_tokens"] if ppt else 0,
               "parameters": sum(p.numel() for p in model.parameters()),
               "core_initial_sha256": ppt["core_initial_sha256"] if ppt else start_core,
               "language_start_core_sha256": start_core, "language_head_initial_sha256": start_head,
               "language_optimizer_initial_state_entries": 0,
               "language_train_order_sha256": order_hash.hexdigest(), "language_schedule_sha256": schedule_hash.hexdigest(),
               "initial_validation_nll": first["validation_nll"], "final_validation_nll": last["validation_nll"],
               "language_train_seconds": train_seconds, "language_seconds": last["language_seconds"],
               "cumulative_seconds": last["cumulative_seconds"], "test": test,
               "checkpoint": "language_final.pt", "checkpoint_sha256": sha_file(folder/"language_final.pt")}
    dump(folder/"language_summary.json", summary)
    return summary


In [ ]:
%%writefile transfer_v4/verify.py
from __future__ import annotations

import json
from pathlib import Path

import torch
import numpy as np
import hashlib

from .common import ARMS, DATA_KEYS, dump, effective_tokens, sha_file, state_sha, source_hashes
from .graph import validate_topology
from .data import load_data
from .model import LM
from .training import evaluate
from .controls import validate_prefix_control
from .plan import verify_plan
from .analysis import common_target_analysis


def verify(root, data_root, device="cpu"):
    root = Path(root)
    env = json.loads((root/"environment.json").read_text())
    c = env["config"]
    verify_plan(root, c)
    if source_hashes() != env["source_sha256"]:
        raise ValueError("Verification source differs from the recorded runner")
    datasets, _ = load_data(data_root, c)
    for seed in c["seeds"]:
        arrays = datasets[str(seed)]
        validate_topology(arrays["graph_edges"], c["nca"]["grid"])
        rule_sets = []
        for split in ("train", "validation"):
            for original, control in (("nca", "prefix_grid"), ("graph", "prefix_graph")):
                validate_prefix_control(arrays[f"{original}_{split}"], arrays[f"{control}_{split}"], c)
            with np.load(Path(data_root)/f"replicate_{seed}/rules_{split}.npz", allow_pickle=False) as rules:
                rule_sets.append({hashlib.sha256(b"".join(rules[k][i].tobytes() for k in sorted(rules.files))).hexdigest()
                                  for i in range(len(rules[rules.files[0]]))})
            for original, shuffled in (("nca", "shuffled"), ("graph", "shuffled_graph")):
                a, b = arrays[f"{original}_{split}"], arrays[f"{shuffled}_{split}"]
                if not np.array_equal(np.sort(a, axis=1), np.sort(b, axis=1)):
                    raise ValueError("Shuffled trajectory histogram changed")
                markers = a >= c["nca"]["states"]**4
                if not np.array_equal(markers, b >= c["nca"]["states"]**4) or not np.array_equal(a[markers], b[markers]):
                    raise ValueError("Shuffled markers changed")
        if rule_sets[0] & rule_sets[1]:
            raise ValueError("Actual train/validation rule weights overlap")
    manifest_hash = sha_file(Path(data_root)/"manifest.json")
    if manifest_hash != env["data_manifest_sha256"]:
        raise ValueError("Data manifest changed")
    records, language = [], {}
    selected_count = 0
    extra_final_count = 0
    initializations = {}
    for seed in c["seeds"]:
        entry = env["initializations"][str(seed)]
        path = root/entry["file"]
        if sha_file(path) != entry["sha256"]:
            raise ValueError("Saved initialization checksum mismatch")
        saved = torch.load(path, map_location="cpu", weights_only=True)
        if saved["config"] != c or saved["seed"] != seed:
            raise ValueError("Saved initialization identity mismatch")
        if state_sha(saved["core"]) != entry["core_sha256"] or state_sha(saved["embedding"]) != entry["head_sha256"]:
            raise ValueError("Saved initialization state mismatch")
        initializations[seed] = entry
    for arm in ARMS:
        for seed in c["seeds"]:
            arrays = datasets[str(seed)]
            folder = root/f"{arm}_{seed}"
            for stage in ("ppt", "language"):
                path = folder/f"{stage}_summary.json"
                if not path.exists():
                    continue
                summary = json.loads(path.read_text())
                checkpoint = folder/summary["checkpoint"]
                if sha_file(checkpoint) != summary["checkpoint_sha256"]:
                    raise ValueError(f"Checkpoint changed: {checkpoint.name}")
                saved = torch.load(checkpoint, map_location=device, weights_only=True)
                if saved["config"] != c or saved["arm"] != arm or saved["seed"] != seed or saved["stage"] != stage:
                    raise ValueError("Checkpoint metadata mismatch")
                if saved["data_manifest_sha256"] != manifest_hash:
                    raise ValueError("Checkpoint/data mismatch")
                model = LM(c["model"], saved["vocab"], core_seed=seed, head_seed=seed)
                model.load_state_dict(saved["model"], strict=True)
                model.to(device)
                if stage == "ppt":
                    key = DATA_KEYS[arm]+"_validation"
                    if saved["core_initial_sha256"] != initializations[seed]["core_sha256"]:
                        raise ValueError("PPT initial backbone differs from saved common initialization")
                    measured = evaluate(model, arrays[key], c, device, True)
                    errors = {k: abs(measured[k]-saved["validation"][k]) for k in ("nll", "patch_nll", "patch_accuracy")}
                else:
                    if summary["language_updates"] != c["language"]["steps"] or summary["language_tokens"] != c["language"]["steps"]*effective_tokens(c):
                        raise ValueError("Language budgets differ")
                    measured = evaluate(model, arrays["text_validation"], c, device, False)
                    errors = {"validation_nll": abs(measured["nll"]-saved["validation"]["nll"])}
                    if saved["test"] is not None:
                        test = evaluate(model, arrays["text_test"], c, device, False)
                        errors["test_nll"] = abs(test["nll"]-saved["test"]["nll"])
                    language[arm, seed] = summary
                if any(v > 1e-4 for v in errors.values()):
                    raise ValueError(f"Reloaded loss differs: {arm}, {seed}, {stage}, {errors}")
                records.append({"arm": arm, "seed": seed, "stage": stage,
                                "checksum_valid": True, "absolute_errors": errors})
                selected_count += 1
                print(f"Verified {stage} {arm} seed={seed}: max error {max(errors.values()):.3g}", flush=True)
                if stage == "ppt" and (folder/"ppt_final.pt").exists():
                    if sha_file(folder/"ppt_final.pt") != summary["final_checkpoint_sha256"]:
                        raise ValueError("Final PPT checkpoint checksum mismatch")
                    final = torch.load(folder/"ppt_final.pt", map_location=device, weights_only=True)
                    if final["config"] != c or final["arm"] != arm or final["seed"] != seed or final["data_manifest_sha256"] != manifest_hash:
                        raise ValueError("Final PPT checkpoint identity mismatch")
                    model.load_state_dict(final["model"], strict=True)
                    measured = evaluate(model, arrays[key], c, device, True)
                    final_errors = {k: abs(measured[k]-final["validation"][k]) for k in ("nll", "patch_nll", "patch_accuracy")}
                    if max(final_errors.values()) > 1e-4:
                        raise ValueError("Final PPT checkpoint loss mismatch")
                    records.append({"arm": arm, "seed": seed, "stage": "ppt_final",
                                    "absolute_errors": final_errors, "sha256": sha_file(folder/"ppt_final.pt")})
                    extra_final_count += 1
    for seed in c["seeds"]:
        summaries = [language[a, seed] for a in ARMS if (a, seed) in language]
        if summaries and len(summaries) != len(ARMS):
            raise ValueError("Incomplete language arm set")
        for key in ("core_initial_sha256", "language_head_initial_sha256", "language_train_order_sha256",
                    "language_schedule_sha256", "language_tokens", "language_updates"):
            if len({s[key] for s in summaries}) > 1:
                raise ValueError(f"Unmatched language control: {key}, seed {seed}")
        if any(s["language_optimizer_initial_state_entries"] != 0 for s in summaries):
            raise ValueError("Language optimizer was not fresh")
        for s in summaries:
            entry = initializations[seed]
            if s["core_initial_sha256"] != entry["core_sha256"] or s["language_head_initial_sha256"] != entry["head_sha256"]:
                raise ValueError("Language common initialization differs from saved tensors")
            if s["arm"] == "scratch":
                expected_core = entry["core_sha256"]
            else:
                ppt = torch.load(root/f"{s['arm']}_{seed}/ppt_selected.pt", map_location="cpu", weights_only=True)
                expected_core = state_sha({k[5:]: v for k, v in ppt["model"].items() if k.startswith("core.")})
            if expected_core != s["language_start_core_sha256"]:
                raise ValueError("Language did not start from the declared PPT core")
    status = json.loads((root/"status.json").read_text())
    expected = len(c["seeds"])*(13 if status["state"] == "complete" else 6)
    if selected_count != expected:
        raise ValueError(f"Expected {expected} selected-stage checkpoints; verified {selected_count}")
    curves = {(arm, seed): [json.loads(line) for line in (root/f"{arm}_{seed}/language_metrics.jsonl").read_text().splitlines()]
              for arm, seed in language}
    steps = [0]+list(range(c["eval_every"], c["language"]["steps"]+1, c["eval_every"]))
    if steps[-1] != c["language"]["steps"]:
        steps.append(c["language"]["steps"])
    for (arm, seed), rows in curves.items():
        if [r["language_step"] for r in rows] != steps:
            raise ValueError("Raw language curve has missing, repeated or extra evaluations")
        if any(r["arm"] != arm or r["seed"] != seed or r["language_tokens"] != r["language_step"]*effective_tokens(c) for r in rows):
            raise ValueError("Raw language curve identity/exposure mismatch")
        if abs(rows[-1]["validation_nll"]-language[arm, seed]["final_validation_nll"]) > 1e-4:
            raise ValueError("Raw curve endpoint differs from the verified checkpoint summary")
    if json.loads((root/"report.json").read_text())["primary_analysis"] != common_target_analysis(curves, c):
        raise ValueError("Primary analysis differs from raw recorded language curves")
    result = {"device": device, "status": status, "stage_checkpoints_verified": len(records),
              "selected_stage_checkpoints_verified": selected_count,
              "additional_final_ppt_checkpoints_verified": extra_final_count,
              "saved_initializations_verified": len(initializations),
              "records": records, "language_controls_checked": bool(language),
              "note": "Checkpoint consistency does not prove learning acceleration"}
    dump(root/"verification.json", result)
    return result


In [ ]:
%%writefile configs/historical_sources_pre_v4.json
{
  "transfer_v2/__init__.py": "491ce87fd9e35cefc0dcf7325216026f4b404099c6182971479adcd06258072e",
  "transfer_v2/calibrate.py": "965fb0fcc0958137f5a08478bb1069092aa8c0ac37b5aa5cf80136eaca030909",
  "transfer_v2/common.py": "24967a682e41f4bf9dfde0add30596a826fa68cbd3fd8a2ca42a40d800c35e9a",
  "transfer_v2/data.py": "55f77791c6fa0927e86ae794c223df49d82fe5410a6a097a01334fc8207d1f4f",
  "transfer_v2/model.py": "4394c7c226ec72113925da5fb9c50949361d652c5d47f54926e399a5b8f8c6b7",
  "transfer_v2/nca.py": "4b7bbf83f70614d2feca5619a68f4313a18172900d0fb3397d14d96f0d0bc52f",
  "transfer_v2/report.py": "537f267f7a613483a57b319aa0ff0051cfc9b287a919805a38863df69f6a0925",
  "transfer_v2/run.py": "8e0f9247c49b66e9656ddb552886e1897f6b529508dac6d2f96f46e7659e565e",
  "transfer_v2/training.py": "fa34eb401bacf01490191bbcfa4729db95e4db2fed95f0de68a15337a36abfe5",
  "transfer_v2/verify.py": "872a54ebb8a24c5ced0eb4bad027e8d01a73cb61033a5295d1d26b913b25a3a1",
  "transfer_v3/__init__.py": "491ce87fd9e35cefc0dcf7325216026f4b404099c6182971479adcd06258072e",
  "transfer_v3/calibrate.py": "3d99fe38477d6a8f01d679bc9ca356de92d7de60ff6d3e3a804ff57f745dcd5b",
  "transfer_v3/common.py": "94c8a870fd3b19b06f7a2d2ccc870c59488a2b596bed7b0c81e572d1d4c1ec95",
  "transfer_v3/data.py": "b3ad2e76dc509934d55f7fb3cd66ed59a388499a0db4bce18b8ff5d76e1e646e",
  "transfer_v3/graph.py": "a0787e8a15f5ff058107260c5f762615cdc832848c2bfd484c927f7860022ee2",
  "transfer_v3/model.py": "4394c7c226ec72113925da5fb9c50949361d652c5d47f54926e399a5b8f8c6b7",
  "transfer_v3/nca.py": "d12642ce78d7826ad9ad4f08a81e53a0c53e32cbf23d78e03074f5cb6942ed66",
  "transfer_v3/report.py": "ee9e69294fd8a7d314ed4541eedf05f2c31e07682762d50fc0220b190a096f05",
  "transfer_v3/run.py": "6badb47576c49f398b82850f4ceda177c54b1d601f692ed02d89fea61e704ad7",
  "transfer_v3/training.py": "de2e03c560b8c51813a7b59de7488a3a2b7d85a88ea33a00bf2fee8177a668fa",
  "transfer_v3/verify.py": "4c738cad63c046975c215b1a0b7ddf34884e26afa7a0331d3b955b53d49b4e05",
  "experiment.py": "76b535acbf08050295648d5692d46690366c1f4802d5668c99edeca003cf1981"
}


In [ ]:
%%writefile configs/transfer_v4.json
{
  "schema_version": 4,
  "purpose": "development",
  "seeds": [
    59,
    71,
    83,
    97,
    109,
    127
  ],
  "precision": "fp32",
  "model": {
    "layers": 6,
    "heads": 4,
    "width": 192,
    "context": 1024
  },
  "batch": 2,
  "accumulate": 2,
  "eval_every": 100,
  "eval_batch": 2,
  "language_eval_tokens": 32768,
  "ppt_eval_rules": 32,
  "ppt": {
    "steps": 1000,
    "lr": 0.0001,
    "warmup_fraction": 0.1,
    "weight_decay": 0.0,
    "clip": null
  },
  "language": {
    "steps": 2000,
    "lr": 0.0005,
    "warmup_fraction": 0.1,
    "weight_decay": 0.0001,
    "clip": 1.0
  },
  "nca": {
    "grid": 12,
    "states": 10,
    "burnin": 10,
    "temperature": 0.0001,
    "profile_frames": 10,
    "filter_low": 0.5,
    "filter_high": 1.0,
    "train_rules": 4096,
    "val_rules": 128,
    "candidate_batch": 64,
    "max_candidates": 262144,
    "data_seed": 20261020,
    "channel_padding": "spatial_only"
  },
  "gate": {
    "enforce": true,
    "minimum_relative_improvement": 0.05,
    "baseline_margin_nats": 0.01
  },
  "text": {
    "dataset": "Salesforce/wikitext",
    "subset": "wikitext-2-raw-v1",
    "revision": "b08601e04326c79dfdd32d625aee71d232d685c3",
    "tokenizer": "gpt2",
    "evaluate_test": false
  },
  "graph": {
    "rewire_fraction": 0.25,
    "topology_seed": 20261019,
    "rule_selection": "intersection",
    "replication": "independent_cache_per_model_seed"
  },
  "study": {
    "id": "prefix_frequency_control_v1",
    "target": "paired_scratch_endpoint",
    "primary_pairs": [
      [
        "grid_nca",
        "graph_nca"
      ],
      [
        "prefix_graph",
        "graph_nca"
      ]
    ],
    "alpha_per_comparison": 0.025,
    "minimum_cohorts": 6,
    "control": "initial_frame_and_supervised_prefix_histogram",
    "crossing": "first_recorded_no_interpolation_all_cohorts_required"
  }
}


In [ ]:
%%writefile configs/transfer_v4_smoke.json
{
  "schema_version": 4,
  "purpose": "execution_check",
  "seeds": [
    17
  ],
  "precision": "fp32",
  "model": {
    "layers": 2,
    "heads": 2,
    "width": 32,
    "context": 76
  },
  "batch": 2,
  "accumulate": 1,
  "eval_every": 2,
  "eval_batch": 2,
  "language_eval_tokens": 304,
  "ppt_eval_rules": 4,
  "ppt": {
    "steps": 4,
    "lr": 0.001,
    "warmup_fraction": 0.25,
    "weight_decay": 0.0,
    "clip": null
  },
  "language": {
    "steps": 6,
    "lr": 0.001,
    "warmup_fraction": 0.2,
    "weight_decay": 0.0001,
    "clip": 1.0
  },
  "nca": {
    "grid": 12,
    "states": 10,
    "burnin": 10,
    "temperature": 0.0,
    "profile_frames": 10,
    "filter_low": 0.0,
    "filter_high": 1.1,
    "train_rules": 8,
    "val_rules": 4,
    "candidate_batch": 8,
    "max_candidates": 128,
    "data_seed": 20261020,
    "channel_padding": "spatial_only"
  },
  "gate": {
    "enforce": false,
    "minimum_relative_improvement": 0.05,
    "baseline_margin_nats": 0.01
  },
  "text": {
    "dataset": "offline_fixture",
    "subset": null,
    "revision": null,
    "tokenizer": "byte_fixture",
    "evaluate_test": false
  },
  "graph": {
    "rewire_fraction": 0.25,
    "topology_seed": 20261019,
    "rule_selection": "intersection",
    "replication": "independent_cache_per_model_seed"
  },
  "study": {
    "id": "prefix_frequency_control_v1",
    "target": "paired_scratch_endpoint",
    "primary_pairs": [
      [
        "grid_nca",
        "graph_nca"
      ],
      [
        "prefix_graph",
        "graph_nca"
      ]
    ],
    "alpha_per_comparison": 0.025,
    "minimum_cohorts": 6,
    "control": "initial_frame_and_supervised_prefix_histogram",
    "crossing": "first_recorded_no_interpolation_all_cohorts_required"
  }
}


In [ ]:
%%writefile tests/test_transfer_v4.py
"""New mechanism/analysis regressions; full training paths have offline fixtures."""
import copy
import hashlib
import json
from pathlib import Path
import unittest

import numpy as np
import torch

from transfer_v4.analysis import common_target_analysis
from transfer_v4.common import ARMS, frame_length, load_config, validate_config
from transfer_v4.controls import prefix_shuffled, validate_prefix_control
from transfer_v4.nca import encode_frames, targets, control_probabilities, shuffled

ROOT = Path(__file__).resolve().parents[1]


class MechanismChecks(unittest.TestCase):
    def setUp(self):
        self.c = load_config(ROOT/'configs/transfer_v4_smoke.json')
        rng = torch.Generator().manual_seed(392)
        self.x = encode_frames(torch.randint(10, (8, 5, 144), generator=rng), 12, 10).numpy().astype(np.uint16)

    def test_prefix_histogram_matches_actual_supervised_targets(self):
        y = prefix_shuffled(self.x, self.c, 42)
        validate_prefix_control(self.x, y, self.c)
        boundary = self.c['model']['context']+1
        a = targets(torch.tensor(self.x[:, :boundary].astype(np.int64)), self.c, patch_only=True).numpy()
        b = targets(torch.tensor(y[:, :boundary].astype(np.int64)), self.c, patch_only=True).numpy()
        np.testing.assert_array_equal(np.sort(a, axis=1), np.sort(b, axis=1))
        pa, _ = control_probabilities(self.x, self.c)
        pb, _ = control_probabilities(y, self.c)
        np.testing.assert_array_equal(pa, pb)
        self.assertFalse(np.array_equal(self.x[:, frame_length(self.c):boundary], y[:, frame_length(self.c):boundary]))

    def test_partial_final_frame_boundary_and_initial_input_counts(self):
        c = copy.deepcopy(self.c)
        c['model']['context'] = 101  # Boundary lies inside a patch frame.
        y = prefix_shuffled(self.x, c, 81)
        validate_prefix_control(self.x, y, c)
        np.testing.assert_array_equal(np.sort(self.x[:, :102], axis=1), np.sort(y[:, :102], axis=1))

    def test_control_is_reproducible_but_seed_changes_order(self):
        a = prefix_shuffled(self.x, self.c, 42)
        np.testing.assert_array_equal(a, prefix_shuffled(self.x, self.c, 42))
        self.assertFalse(np.array_equal(a, prefix_shuffled(self.x, self.c, 43)))
        self.assertEqual(a.dtype, np.uint16)

    def test_no_suffix_tokens_leak_into_training_prefix(self):
        c = self.c
        x = self.x.copy()
        boundary = c['model']['context']+1
        suffix = x[:, boundary:]
        suffix[suffix < 10000] = 9999
        y = prefix_shuffled(x, c, 42)
        self.assertEqual(int((y[:, :boundary] == 9999).sum()), int((x[:, :boundary] == 9999).sum()))

    def test_default_v3_gap_is_masked_conditioning_not_unused_suffix(self):
        c = load_config(ROOT/'configs/transfer_v4.json')
        g = torch.Generator().manual_seed(124)
        x = encode_frames(torch.randint(10, (4, 27, 144), generator=g), 12, 10).numpy().astype(np.uint16)
        old = shuffled(x, c, 123)
        boundary = c['model']['context']+1
        self.assertEqual(x.shape[1], boundary+1)
        self.assertTrue(np.all(x[:, boundary:] >= 10000))
        np.testing.assert_array_equal(np.sort(x[:, :boundary], axis=1), np.sort(old[:, :boundary], axis=1))
        a = targets(torch.tensor(x[:, :boundary].astype(np.int64)), c, patch_only=True).numpy()
        b = targets(torch.tensor(old[:, :boundary].astype(np.int64)), c, patch_only=True).numpy()
        self.assertFalse(np.array_equal(np.sort(a, axis=1), np.sort(b, axis=1)))
        self.assertFalse(np.array_equal(x[:, :frame_length(c)], old[:, :frame_length(c)]))
        validate_prefix_control(x, prefix_shuffled(x, c, 123), c)

    def test_initial_frame_markers_and_histogram_corruption_rejected(self):
        good = prefix_shuffled(self.x, self.c, 42)
        for index in (1, 38, 39):
            with self.subTest(index=index):
                bad = good.copy()
                bad[0, index] = (int(bad[0, index])+1) % 10000
                with self.assertRaises(ValueError):
                    validate_prefix_control(self.x, bad, self.c)

    def test_short_sequence_rejected(self):
        with self.assertRaises(ValueError):
            prefix_shuffled(self.x[:, :30], self.c, 42)

    def test_default_is_six_fresh_cohorts_and_seven_arms(self):
        c = load_config(ROOT/'configs/transfer_v4.json')
        self.assertEqual(len(c['seeds']), 6)
        self.assertFalse(set(c['seeds']) & {17, 29, 43})
        self.assertEqual(len(ARMS), 7)
        for mutation in ('criterion', 'old_seed', 'few_seeds', 'test'):
            bad = copy.deepcopy(c)
            if mutation == 'criterion': bad['study']['alpha_per_comparison'] = .05
            elif mutation == 'old_seed': bad['seeds'][0] = 43
            elif mutation == 'few_seeds': bad['seeds'] = bad['seeds'][:3]
            else: bad['text']['evaluate_test'] = True
            with self.subTest(mutation=mutation), self.assertRaises(ValueError):
                validate_config(bad)

    def test_frozen_sources_and_mechanistic_identity(self):
        manifest = ROOT/'configs/historical_sources_pre_v4.json'
        if not (ROOT/'transfer_v3').exists():
            self.skipTest('Historical packages are not embedded in the standalone v4 notebook')
        for name, expected in json.loads(manifest.read_text()).items():
            self.assertEqual(hashlib.sha256((ROOT/name).read_bytes()).hexdigest(), expected, name)
        for name in ('model.py', 'nca.py', 'graph.py', 'training.py'):
            # Training only changes checkpoint schema; numerical updates are fixed.
            a = (ROOT/'transfer_v3'/name).read_text().replace('"schema_version": 3', '"schema_version": 4')
            self.assertEqual(a, (ROOT/'transfer_v4'/name).read_text(), name)
        old = json.loads((ROOT/'configs/transfer_v3.json').read_text())
        new = load_config(ROOT/'configs/transfer_v4.json')
        for key in old:
            if key not in ('schema_version', 'seeds'):
                self.assertEqual(old[key], new[key], key)


class AnalysisChecks(unittest.TestCase):
    def fixture(self):
        c = load_config(ROOT/'configs/transfer_v4.json')
        def row(step, nll):
            return {'language_step': step, 'validation_nll': nll, 'language_tokens': step*4096,
                    'language_seconds': step+1., 'cumulative_seconds': step+1000.}
        curves = {}
        for seed in c['seeds']:
            curves['scratch', seed] = [row(0, 10), row(2000, 5)]
            for arm in ('grid_nca', 'prefix_graph'):
                curves[arm, seed] = [row(0, 10), row(1800, 5), row(2000, 4.9)]
            curves['graph_nca', seed] = [row(0, 10), row(1600, 5), row(2000, 4.8)]
        return c, curves

    def test_six_wins_pass_both_fixed_tests_at_common_target(self):
        c, curves = self.fixture()
        a = common_target_analysis(curves, c)
        self.assertTrue(a['supports_graph_advantage_on_declared_target'])
        for item in a['comparisons']:
            self.assertEqual(item['one_sided_sign_p'], 1/64)
            self.assertAlmostEqual(item['geomean_language_tokens_speedup_all_cohorts'], 1800/1600)
            self.assertEqual({r['target_nll'] for r in item['rows']}, {5})

    def test_one_tie_is_not_dropped_and_prevents_claim(self):
        c, curves = self.fixture()
        curves['grid_nca', c['seeds'][-1]] = copy.deepcopy(curves['graph_nca', c['seeds'][-1]])
        a = common_target_analysis(curves, c)
        self.assertFalse(a['supports_graph_advantage_on_declared_target'])
        self.assertEqual(a['comparisons'][0]['one_sided_sign_p'], 7/64)
        self.assertEqual(len(a['comparisons'][0]['rows']), 6)

    def test_censoring_withholds_aggregate_and_significance(self):
        c, curves = self.fixture()
        for row in curves['graph_nca', c['seeds'][-1]]:
            row['validation_nll'] = 8.
        a = common_target_analysis(curves, c)
        self.assertFalse(a['supports_graph_advantage_on_declared_target'])
        for item in a['comparisons']:
            self.assertIsNone(item['one_sided_sign_p'])
            self.assertIsNone(item['geomean_language_tokens_speedup_all_cohorts'])

    def test_missing_cohort_and_zero_step_are_undefined(self):
        for kind in ('missing', 'zero', 'truncated_scratch'):
            c, curves = self.fixture()
            if kind == 'missing': del curves['scratch', c['seeds'][0]]
            elif kind == 'truncated_scratch': curves['scratch', c['seeds'][0]][-1]['language_step'] = 1900
            else: curves['graph_nca', c['seeds'][0]][0]['validation_nll'] = 4.
            a = common_target_analysis(curves, c)
            self.assertFalse(a['supports_graph_advantage_on_declared_target'])
            self.assertIsNone(a['comparisons'][0]['geomean_language_tokens_speedup_all_cohorts'])

    def test_fixture_cannot_support_research_claim(self):
        c, curves = self.fixture()
        c['purpose'] = 'execution_check'
        self.assertFalse(common_target_analysis(curves, c)['supports_graph_advantage_on_declared_target'])


if __name__ == '__main__':
    unittest.main()


In [ ]:
%%writefile requirements.txt
numpy>=1.26,<3
torch>=2.5,<3
matplotlib>=3.8,<4


In [ ]:
%%writefile requirements-v2.txt
-r requirements.txt
tiktoken>=0.7,<1
pyarrow>=17,<27


In [ ]:
%%writefile docs/TRANSFER_V4_PROTOCOL.md
# V4: prospective replication and exact-prefix controls

Prepared 19 October 2026, after inspecting v3. This is a new validation-stage
replication/mechanism study. It does not retroactively change v3's primary
grid-endpoint result, and it is not confirmation on an untouched language task.
The objective remains **acceleration of language learning through pre-pre-training**.
The graph NCA supplies synthetic PPT trajectories; it is not the language model.

## Frozen treatment and budgets

Use `configs/transfer_v4.json` with six new paired cohort seeds
**59, 71, 83, 97, 109, 127**. Each seed changes model initialization, synthetic
rule bank and topology together, just as in v3. These variance sources are not
separately estimated. No inspected v3 seed (17, 29, 43) enters this cohort.

Seven arms enter the identical language stage:

| Arm | PPT data | PPT updates | Language updates |
| --- | --- | ---: | ---: |
| scratch | None | 0 | 2000 |
| grid_nca | Grid trajectories | 1000 | 2000 |
| graph_nca | Rewired graph trajectories | 1000 | 2000 |
| shuffled_nca | V3 whole-trajectory grid shuffle | 1000 | 2000 |
| shuffled_graph | V3 whole-trajectory graph shuffle | 1000 | 2000 |
| prefix_grid | New exact-prefix grid shuffle | 1000 | 2000 |
| prefix_graph | New exact-prefix graph shuffle | 1000 | 2000 |

Preserve v3's transformer, 4096 input tokens/update, 1024 context, tokenizer,
WikiText revision/splits, optimizer settings, 100-update evaluation cadence,
32 held-out synthetic evaluation rules, rule counts, compression band, burn-in,
25% ordered-slot graph rewiring, joint grid/graph rule admission and fixed-node
2x2 serializer. Do not introduce graph-local serialization or tune topology here.
Grid/graph still share frozen rule weights and rollout random draws. Actual common
starting backbones and language heads are saved; each language optimizer starts
fresh, with matched data order, scheduler and full-backbone transfer.

Both existing shuffled arms remain to connect the new mechanism comparison to
v3. V3 results themselves remain immutable. Do not pool v2/v3 with these cohorts.

## Stronger frequency/format control

For **each individual trajectory**, leave the entire first conditioning frame
and every start/end marker unchanged. Uniformly permute the subsequent patch
tokens **within the exact prefix used by training and evaluation**, separately
from any unused suffix. The exclusive prefix boundary is `context + 1` = 1025:
next-token labels include the token at absolute index 1024. Marker positions
and the initial-context mask are identical across a parent and its control.

This matches both the full trained input-plus-target prefix histogram and its
supervised patch-target histogram exactly, per rule. It also preserves the
initial conditioning frame and therefore its histogram. In the default v3
layout, the 1026-token trajectory has only a final end marker outside the
1025-token input-plus-target prefix: **the old shuffle already preserves that
prefix's overall histogram**. Its actual mismatch is exchanging patches with
the masked initial frame, changing both conditioning and supervised-target
frequencies. The new control fixes that mismatch. Separate suffix shuffling is
a general safeguard when a configuration has unused suffix patches; it does
not repair a demonstrated suffix-leakage bug in the default v3 run.
No fresh rule filtering or
selection is performed for any shuffled arm. Grid and graph use the same
permutation seed within a cohort/split; train and validation use distinct seeds.

This control destroys patch ordering while retaining rule-specific frequencies,
formatting and conditioning/frequency associations. It does **not** isolate all
possible physical mechanisms, preserve per-frame histograms, or eliminate all
statistical dependence. A graph-versus-prefix benefit would support an effect
of order-dependent structure beyond these matched quantities, not prove that
local interaction physics is the unique cause. Native PPT loss alone is not a
cross-dataset measure of dynamics learning.

## Admission gate and stopping

Every structured grid and graph PPT model must improve held-out patch NLL by
at least 5% from initialization and beat both training-fitted unigram and
previous-frame-copy mixture controls by at least 0.01 nats. The selected PPT
checkpoint minimizes held-out patch NLL; charge all 1000 updates regardless.
Shuffled gates are diagnostics, not admission requirements. Any structured
failure stops the entire language suite, exports diagnostics and retains every
cohort. No automatic retry, rule-threshold relaxation, seed replacement or
post-result budget/target change is allowed. Tiny byte fixtures can bypass the
gate only under `purpose=execution_check` and cannot support a research claim.

## Prospectively fixed comparisons

For each new cohort, define **one common target**: that cohort's scratch model's
final validation NLL at 2000 language updates. Find each arm's first recorded
crossing at or below that target, including step zero, with no interpolation.
This replaces v3's different reference-arm endpoints **only for this new study**.
It avoids comparing graph/grid and graph/control speed at different losses.

The two co-primary comparisons are:

1. graph_nca versus grid_nca;
2. graph_nca versus prefix_graph.

For each, report the paired ratio `reference tokens / graph tokens` at the
same scratch target. Ratios above one favor graph. Publish all six rows, strict
wins/ties, geometric mean, and a one-sided exact sign test. Ties count as
non-wins, conservatively. Use alpha **0.025 per comparison** (Bonferroni family
bound 0.05 for the two tests). Both tests must pass and both geometric means
must exceed one before describing support for a graph advantage on this target.
With six cohorts this stringent rule requires six strict wins in each comparison:
`p = 1/64 = 0.015625`; five wins yields `p = 7/64 = 0.109375`.
This design detects consistent effects and can remain inconclusive despite a
positive mean; it is not a guarantee of adequate power for small effects.

If either arm misses the target in any cohort, retain that row as censored and
withhold the all-cohort ratio and sign-test claim. Missing cohorts are not wins.
A zero-token crossing makes a token ratio undefined; do not turn it into an
infinite speedup. The independent test unit is the paired cohort, not a batch,
checkpoint, validation token or bootstrap draw. Six cohorts do not establish
generalization to larger models, new graph families or other language tasks.

Report all arms versus scratch, grid versus prefix_grid, original shuffled
controls, final NLL/perplexity and the historical reference-endpoint comparisons
as secondary/descriptive outcomes. A graph advantage over grid/control does not
by itself establish a benefit over scratch; report that crossing separately.
An eventual confirmation experiment must reserve new data and its own criterion.
The official language test is neither downloaded nor evaluated in v4 defaults.

## Timing, evidence and execution

Report language-token savings as the primary learning-speed measurement.
Report language-stage wall time and cumulative generation + PPT + language
time separately, at the same target, retaining censored cohorts. Cumulative
cost conservatively charges the whole paired preparation to each synthetic arm;
it is not an isolated generator throughput benchmark or cloud-billing measure.
Retain v3's detailed timing exclusions in `environment.json`. Model/data
preparation and all checkpoint verification add to actual notebook runtime.

V4 runs 36 PPT jobs and 42 language jobs (78 selected stage checkpoints and
six actual initialization snapshots). Scaling the measured v3 stage workload
suggests roughly **7-8 hours on a comparable T4**, with variable preparation,
verification and export overhead beyond that estimate. This is an extrapolation,
not a new GPU measurement or service quota guarantee. The notebook prints a
disposable timing calibration on the actual assigned device before training.
Do not shrink cohorts or stages after inspecting new outcomes and still call
the result this frozen protocol; declare a new development version instead.

`analysis_plan.json`, its checksum in the run environment, and an archived
`FROZEN_PROTOCOL.md` are saved before benchmark model initialization/training.
They retain the config, study criterion and runner hashes. The verifier checks
plan/source/data identity, prefix-control invariants, topology/rule separation,
all selected/final checkpoint losses, actual initialization/transfer hashes,
matched language controls and primary analysis recomputed from raw curves.
Keep original ZIPs and all failed/stopped workflows. Completion is engineering
success, not evidence that acceleration occurred.

Upload `kaggle_nca_transfer_v4.ipynb`, enable one GPU and Internet, and Run All.
The last cell prints `WORKFLOW FINISHED: complete` or
`WORKFLOW FINISHED: stopped_at_synthetic_gate` and exports
`nca_transfer_v4_results_<timestamp>.zip`. No remote training is launched by
preparing this notebook locally. CPU checks only verify implementation.


In [ ]:
%%writefile LICENSE

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.


In [ ]:
%%writefile NOTICE
GNNs in NCA — graph cellular automata pre-pre-training research
Copyright 2026 GNNs in NCA contributors

Original project code and documentation are licensed under Apache License 2.0.
See LICENSE. This notice does not modify any license terms.

The pinned source under sources/nca-pre-pretraining-reference/ is
Copyright (c) 2026 Dan Lee and remains under its included MIT license.
It is reference material, not a runtime dependency or an assertion of endorsement.

Tiny Shakespeare is downloaded or supplied at runtime, not included here.
Dependencies are distributed separately under their respective licenses.


In [ ]:
%%writefile THIRD_PARTY_NOTICES.md
# Third-party material

The unchanged snapshot in `sources/nca-pre-pretraining-reference/` comes from
[danihyunlee/nca-pre-pretraining](https://github.com/danihyunlee/nca-pre-pretraining),
revision `bdd1c71e04eb2e5a2c91481faf278e95846b4431` (12 March 2026).
Copyright (c) 2026 Dan Lee. It remains under the included
[MIT license](sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431/LICENSE).
`reference.json` records URLs/hashes. It is not imported by the pilot; its extra
dependencies are not required for this project's checks.

The methodology audit discusses
[Training Language Models via Neural Cellular Automata, v1](https://arxiv.org/html/2603.10055v1).
The paper and competition rules PDF are not redistributed here.

The default corpus is fetched from the
[Tiny Shakespeare source](https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt)
at runtime. A data manifest records its SHA-256 and byte splits. Review the source's
terms before redistributing derived data bundles. Raw corpus bytes, cached arrays
and model checkpoints are excluded from this source snapshot.

NumPy, PyTorch and Matplotlib are installed separately under their own licenses.
Version 2 additionally uses separately installed tiktoken and PyArrow. Its NCA
port is based on the retained MIT reference, with the explicit changes recorded
in `docs/TRANSFER_V2_PROTOCOL.md`; the standalone notebook includes that MIT
license. WikiText-2 is downloaded at runtime from the pinned Salesforce dataset
revision and retains its CC-BY-SA-3.0/GFDL terms. Dataset/checkpoint artifacts
remain excluded from the public source selection.
`docs/results/` contains project measurements and plots; its provenance manifest
distinguishes copied originals from adapted summaries.


In [ ]:
%%writefile sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431/LICENSE
MIT License

Copyright (c) 2026 Dan Lee

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


## Dependencies and controls
Kaggle supplies its CUDA PyTorch build; this cell retains it and installs only
the small tokenizer/Parquet reader if missing. A GPU is required for the actual
development run. CPU tests are engineering checks, not acceleration evidence.


In [ ]:
import importlib.metadata
from packaging.specifiers import SpecifierSet
missing = []
for package, bounds in [('tiktoken', '>=0.7,<1'), ('pyarrow', '>=17,<27')]:
    try:
        suitable = importlib.metadata.version(package) in SpecifierSet(bounds)
    except importlib.metadata.PackageNotFoundError:
        suitable = False
    if not suitable:
        missing.append(package + bounds)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator and restart the notebook runtime.'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))

def run(*arguments):
    subprocess.run([sys.executable, *arguments], check=True)

run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_transfer_v4.py', '-v')


## Tiny GPU execution fixture
This checks CUDA execution, stage checkpoints, common language controls and
verification. It deliberately uses a tiny byte fixture and does not supply a
scientific result or count toward any development model's training.


In [ ]:
fixture = Path('fixture')
fixture.mkdir()
(fixture / 'train.txt').write_text('Offline execution fixture; no research inference.\n' * 512)
(fixture / 'validation.txt').write_text('Distinct validation fixture for checks.\n' * 128)
run('-m', 'transfer_v4.run', 'all', '--config', 'configs/transfer_v4_smoke.json',
    '--data', 'data/smoke', '--out', 'runs/smoke', '--fixture-dir', str(fixture), '--device', 'cuda')


## Real development run
Data, budgets, selection and gate settings are fixed in `configs/transfer_v4.json`.
The CLI prepares filtered rule-disjoint NCA data, exact-prefix controls and pinned WikiText/GPT-2 tokens,
prints an estimate measured on disposable models on your actual GPU, runs PPT,
then conditionally runs the matched language stage and verifies checkpoints.

The synthetic gate requires at least 5% held-out patch-loss improvement and a
0.01-nat advantage over both fitted unigram/copy controls for every GRID AND GRAPH seed.
It is never relaxed automatically. Test data are neither downloaded nor evaluated.


In [ ]:
run('-m', 'transfer_v4.run', 'all', '--config', 'configs/transfer_v4.json',
    '--data', 'data/development', '--out', 'runs/development', '--device', 'cuda')


## Inspect the outcome
If status is `complete`, read `primary_analysis` first: graph versus grid and
graph versus prefix_graph at each cohort's scratch endpoint. Ratios use the
same target; any missing/zero-token crossing withholds the aggregate. All six
cohorts, ties and censored rows remain visible. Native shuffled controls and
reference-arm endpoints are secondary. Language tokens/time and cumulative
cost are separate measurements; token gains need not repay PPT overhead.

If status is `stopped_at_synthetic_gate`, outputs contain diagnostics, not a
downstream acceleration result. The frozen protocol and plan are in the ZIP.


In [ ]:
import json
from IPython.display import SVG, display

OUT = Path('runs/development')
status = json.loads((OUT / 'status.json').read_text())
print('RUN STATUS:', status['state'])
display(json.loads((OUT / 'ppt_gate.json').read_text()))
result = json.loads((OUT / 'report.json').read_text())
display(result['primary_analysis'])
display(result['aggregate'])
display(result['treatment_comparisons'])
display(json.loads((OUT / 'verification.json').read_text()))
display(SVG(filename=str(OUT / 'ppt_diagnostics.svg')))
if (OUT / 'learning_curves.svg').exists():
    display(SVG(filename=str(OUT / 'learning_curves.svg')))


## Download the verifiable result
Save a Kaggle notebook version with outputs, and download this ZIP. A stopped
synthetic gate is a completed diagnostic workflow, not an error or a successful
transfer claim. It is still useful to inspect and share that output archive.


In [ ]:
import zipfile
from IPython.display import FileLink

archive = Path('/kaggle/working') / f'nca_transfer_v4_results_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    for path in ROOT.rglob('*'):
        if path.is_file() and not {'__pycache__', 'plot_cache'}.intersection(path.relative_to(ROOT).parts):
            z.write(path, path.relative_to(ROOT))
print('WORKFLOW FINISHED:', status['state'])
print('Download:', archive, f'({archive.stat().st_size / 1e6:.1f} MB)')
display(FileLink(str(archive)))
